<a href="https://colab.research.google.com/github/sulucay01/multimodal-rs-segmentation/blob/dev/notebooks/04_multimodal.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 04 — Multimodal experiments

DI725 Term Project. Multimodal Fusion for Remote Sensing Land Cover Segmentation

This notebook trains text-augmented variants of UNetFormer (CLIP-based cross-attention fusion) and addresses the project's main research question along with two ablation studies:

**Main RQ:** Does textual information improve segmentation over the vision-only baseline?

**Caption-variant ablation:** Which caption type (hybrid, text-only, vision-only) gives the largest contribution to segmentation performance?

**Auxiliary regularizer ablation:** Does adding a KL-divergence based train-time auxiliary regularizer provide additional benefit on top of textual augmentation? The auxiliary signal is used only during training and not provided to the model during inference.

A separate methodological experiment also compares caption variants with and without numerical percentages, testing whether the multimodal benefit derives from numerical class proportions or from qualitative semantic content.

All experiments use pixel-level mIoU as the primary metric. The vision-only baseline (UNetFormer, no text) is established in `03_baseline_models.ipynb`. Multimodal experiments here use the same training setup (30 epochs, AdamW lr=6e-4, weighted CE + 0.4 × auxiliary head loss, batch size 8) and only differ in the addition of text fusion.

Sections:
1. Setup
2. Dataset and DataLoaders (with caption)
3. UNetFormer architecture
4. Multimodal architecture (CLIP + cross-attention)
5. Training infrastructure
6. Caption-variant ablation
7. Percentage-removal check
8. KL-divergence auxiliary loss ablation

## 1. Setup

In [1]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
# Install dependencies. open_clip_torch provides the frozen CLIP text encoder.
!pip install transformers timm einops wandb open_clip_torch -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 77.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 4.3 MB/s eta 0:00:00


In [3]:
# All imports for the notebook.
import numpy as np
import pandas as pd
from PIL import Image
import time
import json
import re
import random
from pathlib import Path
import shutil

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

import timm
from timm.layers import DropPath, trunc_normal_
from einops import rearrange
import open_clip

import wandb

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
random.seed(SEED)

Device: cuda


In [ ]:
# Project paths (Drive) and a local working copy for fast I/O during training
PROJECT_DIR     = Path('/content/drive/MyDrive/DI725_Project')
DATA_DIR_DRIVE  = PROJECT_DIR / 'data'
SPLITS_CSV      = DATA_DIR_DRIVE / 'captions_with_splits.csv'
CHECKPOINTS_DIR = PROJECT_DIR / 'checkpoints'
RESULTS_DIR     = PROJECT_DIR / 'results'
CHECKPOINTS_DIR.mkdir(exist_ok=True)
RESULTS_DIR.mkdir(exist_ok=True)

# Local SSD copy avoids Drive I/O bottleneck during training
LOCAL_DATA  = Path('/content/data')
LOCAL_IMAGES      = LOCAL_DATA / 'images'
LOCAL_MASKS_CLASS = LOCAL_DATA / 'masks_class'

if not LOCAL_DATA.exists():
    print('Copying data to local SSD...')
    shutil.copytree(DATA_DIR_DRIVE / 'images', LOCAL_IMAGES)
    shutil.copytree(DATA_DIR_DRIVE / 'masks_class', LOCAL_MASKS_CLASS)
    print('Done.')
else:
    print('Local data already present.')

print(f'Images: {LOCAL_IMAGES}')
print(f'Masks : {LOCAL_MASKS_CLASS}')

Copying data to local SSD...


In [ ]:
# Weights & Biases setup for experiment tracking
wandb.login()

WANDB_PROJECT = 'di725_project'

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results
wandb: Enter your choice:

 2


wandb: You chose 'Use an existing W&B account'
wandb: Logging into https://api.wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Create a new API key at: https://wandb.ai/authorize?ref=models
wandb: Store your API key securely and do not share it.
wandb: Paste your API key and hit enter:

 ··········


wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: selingoktas98 (selingoktas98-metu-middle-east-technical-university) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


In [ ]:
# Class definitions
CLASS_NAMES = ['Tree', 'Shrub', 'Grass', 'Crop', 'Built-up', 'Barren', 'Water']
NUM_CLASSES = len(CLASS_NAMES)

# 5 caption variants from the dataset
CAPTION_COLS = [
    'hybrid_gemma3-4b',
    'hybrid_qwen3-vl-8b',
    'text_qwen3-4b',
    'vision_gemma3-4b',
    'vision_qwen3-vl-8b',
]

In [ ]:
# Load split CSV from Drive
split_df = pd.read_csv(SPLITS_CSV)
train_df = split_df[split_df['split'] == 'train'].reset_index(drop=True)
val_df   = split_df[split_df['split'] == 'val'].reset_index(drop=True)
test_df  = split_df[split_df['split'] == 'test'].reset_index(drop=True)
print(f'Train: {len(train_df)}  |  Val: {len(val_df)}  |  Test: {len(test_df)}')

# Class weights via median frequency balancing (computed from training split)
class_avg = train_df[CLASS_NAMES].mean().values
class_freq = class_avg / class_avg.sum()
median_freq = np.median(class_freq)
class_weights = median_freq / (class_freq + 1e-8)
class_weights_tensor = torch.FloatTensor(class_weights).to(device)
print(f'Class weights: {class_weights.round(2)}')

Train: 7000  |  Val: 1500  |  Test: 1500
Class weights: [0.15 5.12 0.09 0.24 3.77 1.   2.54]


In [ ]:
# Create no-percentage versions of text-rich caption variants.
# These columns are used to test whether the multimodal benefit
# comes from numerical class proportions or from qualitative semantic content.
def remove_percentages(text):
    """Remove numerical percentages from caption while preserving semantic content."""
    if not isinstance(text, str):
        return text
    # Parenthetical numerical expressions: (92%), (approximately 92%)
    text = re.sub(r'\s*\([^)]*\d+[^)]*\)', '', text)
    # "covering 92% of the area"
    text = re.sub(r'\b\d+\.?\d*\s*%\s*of\s+the\s+area\s*,?\s*', '', text, flags=re.IGNORECASE)
    # "at 52%." "by 44%."
    text = re.sub(r'\b(at|by)\s+\d+\.?\d*\s*%\s*\.?', '', text, flags=re.IGNORECASE)
    # Standalone "92%"
    text = re.sub(r'\b\d+\.?\d*\s*%', '', text)
    # "X percent"
    text = re.sub(r'\b(approximately|around|about|nearly|roughly)?\s*\d+\.?\d*\s*percent\b',
                  '', text, flags=re.IGNORECASE)
    # Standalone numbers (with optional preceding qualifier)
    text = re.sub(r'\b(approximately|around|about|nearly|roughly)?\s*\d+\.?\d*\b',
                  '', text, flags=re.IGNORECASE)
    # Broken structure cleanup
    text = re.sub(r'\bwith\s+consisting\s+of\s+', 'with ', text, flags=re.IGNORECASE)
    text = re.sub(r'\bcovering\s+of\s+', 'covering ', text, flags=re.IGNORECASE)
    text = re.sub(r',\s*of\s+', ', ', text, flags=re.IGNORECASE)
    text = re.sub(r'\band\s+of\s+', 'and ', text, flags=re.IGNORECASE)
    # Whitespace cleanup
    text = re.sub(r'\s+', ' ', text).strip()
    text = re.sub(r'\s+,', ',', text)
    text = re.sub(r',\s*,', ',', text)
    text = re.sub(r',\s*\.', '.', text)
    text = re.sub(r'\s+\.', '.', text)
    text = re.sub(r'\.\s*\.', '.', text)
    return text

# Add no-percentage versions for ablation experiment
NOPCT_CAPTIONS = ['text_qwen3-4b', 'hybrid_gemma3-4b', 'hybrid_qwen3-vl-8b']

for split_df in [train_df, val_df, test_df]:
    for col in NOPCT_CAPTIONS:
        split_df[f'{col}_nopct'] = split_df[col].apply(remove_percentages)

# Verify cleaning on a few samples for each variant
for col in NOPCT_CAPTIONS:
    print(f"\n{'=' * 80}")
    print(f"Cleaning verification: {col}_nopct")
    print('=' * 80)
    for i in [0, 5, 10]:
        orig  = train_df.iloc[i][col]
        clean = train_df.iloc[i][f'{col}_nopct']
        print(f"\n--- Sample {i} ---")
        print(f"ORIG : {orig}")
        print(f"CLEAN: {clean}")


Cleaning verification: text_qwen3-4b_nopct

--- Sample 0 ---
ORIG : The scene is predominantly covered by grass (87%), indicating a large area of natural or managed pasture, with minimal presence of crops, built-up areas, or bare land.
CLEAN: The scene is predominantly covered by grass, indicating a large area of natural or managed pasture, with minimal presence of crops, built-up areas, or bare land.

--- Sample 5 ---
ORIG : The scene is predominantly grassland, with a minor presence of trees, indicating an open, natural or agricultural landscape with limited forest cover.
CLEAN: The scene is predominantly grassland, with a minor presence of trees, indicating an open, natural or agricultural landscape with limited forest cover.

--- Sample 10 ---
ORIG : The scene is predominantly agricultural, with 96% of the area covered by crops, indicating a large-scale farming landscape. Grass covers a small fraction (2%), and barren land accounts for another 2%, suggesting minimal natural or unu

## 2. Dataset and DataLoaders

The Dataset returns (image, mask, caption) tuples. The caption column is a constructor argument so the same class works for any of the 5 caption variants.

In [ ]:
# Dataset class — reads pre-converted class-index masks and a caption column
class RSMultiModalDataset(Dataset):
    def __init__(self, dataframe, images_dir, masks_dir, caption_col):
        self.df = dataframe.reset_index(drop=True)
        self.images_dir = Path(images_dir)
        self.masks_dir  = Path(masks_dir)
        self.caption_col = caption_col
        self.normalize = transforms.Normalize(
            mean=[0.485, 0.456, 0.406],
            std=[0.229, 0.224, 0.225],
        )

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        fname = row['filename']

        img = Image.open(self.images_dir / fname).convert('RGB')
        img = transforms.functional.to_tensor(img)
        img = self.normalize(img)

        mask = np.array(Image.open(self.masks_dir / fname))
        mask = torch.from_numpy(mask).long()

        caption = str(row[self.caption_col])
        return img, mask, caption

In [ ]:
# DataLoader factory — produces train/val/test loaders for any caption column.
BATCH_SIZE  = 8
NUM_WORKERS = 4

def make_loaders(caption_col):
    """Build train/val/test loaders for a given caption column."""
    train_dataset = RSMultiModalDataset(train_df, LOCAL_IMAGES, LOCAL_MASKS_CLASS, caption_col)
    val_dataset   = RSMultiModalDataset(val_df,   LOCAL_IMAGES, LOCAL_MASKS_CLASS, caption_col)
    test_dataset  = RSMultiModalDataset(test_df,  LOCAL_IMAGES, LOCAL_MASKS_CLASS, caption_col)

    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,
                              num_workers=NUM_WORKERS, pin_memory=True)
    val_loader   = DataLoader(val_dataset,  batch_size=BATCH_SIZE, shuffle=False,
                              num_workers=NUM_WORKERS, pin_memory=True)
    test_loader  = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False,
                              num_workers=NUM_WORKERS, pin_memory=True)
    return train_loader, val_loader, test_loader

# Sanity check with the first caption variant
train_loader, val_loader, test_loader = make_loaders(CAPTION_COLS[0])
print(f'Caption column: {CAPTION_COLS[0]}')
print(f'Train: {len(train_loader.dataset)} samples, {len(train_loader)} batches')

imgs, masks, captions = next(iter(train_loader))
print(f'Batch: images {imgs.shape}, masks {masks.shape}, captions[0]: "{captions[0][:80]}..."')

Caption column: hybrid_gemma3-4b
Train: 7000 samples, 875 batches
Batch: images torch.Size([8, 3, 256, 256]), masks torch.Size([8, 256, 256]), captions[0]: "The image depicts a landscape dominated by extensive grasslands (81%) interspers..."


## 3. UNetFormer architecture

Same UNetFormer as in `03_baseline_models.ipynb` (CNN encoder `swsl_resnet18` with Global-Local Transformer Block decoder, Feature Refinement Head, and auxiliary head for deep supervision). Re-defined here so this notebook is self-contained.

Source: https://github.com/WangLibo1995/GeoSeg

In [ ]:
# Building blocks (Conv + BN + ReLU variants)
class ConvBNReLU(nn.Sequential):
    """Conv2d then BatchNorm then ReLU6."""
    def __init__(self, in_channels, out_channels, kernel_size=3, dilation=1, stride=1,
                 norm_layer=nn.BatchNorm2d, bias=False):
        super().__init__(
            nn.Conv2d(in_channels, out_channels, kernel_size=kernel_size, bias=bias,
                      dilation=dilation, stride=stride,
                      padding=((stride - 1) + dilation * (kernel_size - 1)) // 2),
            norm_layer(out_channels), nn.ReLU6())


class ConvBN(nn.Sequential):
    """Conv2d then BatchNorm (no activation)."""
    def __init__(self, in_channels, out_channels, kernel_size=3, dilation=1, stride=1,
                 norm_layer=nn.BatchNorm2d, bias=False):
        super().__init__(
            nn.Conv2d(in_channels, out_channels, kernel_size=kernel_size, bias=bias,
                      dilation=dilation, stride=stride,
                      padding=((stride - 1) + dilation * (kernel_size - 1)) // 2),
            norm_layer(out_channels))


class Conv(nn.Sequential):
    """Standalone Conv2d, no normalization or activation."""
    def __init__(self, in_channels, out_channels, kernel_size=3, dilation=1, stride=1, bias=False):
        super().__init__(
            nn.Conv2d(in_channels, out_channels, kernel_size=kernel_size, bias=bias,
                      dilation=dilation, stride=stride,
                      padding=((stride - 1) + dilation * (kernel_size - 1)) // 2))


class SeparableConvBN(nn.Sequential):
    """Depthwise separable convolution: depthwise then BatchNorm then pointwise (1x1)."""
    def __init__(self, in_channels, out_channels, kernel_size=3, stride=1, dilation=1,
                 norm_layer=nn.BatchNorm2d):
        super().__init__(
            nn.Conv2d(in_channels, in_channels, kernel_size, stride=stride, dilation=dilation,
                      padding=((stride - 1) + dilation * (kernel_size - 1)) // 2,
                      groups=in_channels, bias=False),
            norm_layer(out_channels),
            nn.Conv2d(in_channels, out_channels, kernel_size=1, bias=False))

/usr/local/lib/python3.12/dist-packages/timm/models/layers/__init__.py:49: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)


In [ ]:
# MLP and Global-Local Attention (window self-attention + local conv branch)
class Mlp(nn.Module):
    """Two-layer MLP using 1x1 convolutions, used inside transformer blocks."""
    def __init__(self, in_features, hidden_features=None, out_features=None,
                 act_layer=nn.ReLU6, drop=0.):
        super().__init__()
        out_features = out_features or in_features
        hidden_features = hidden_features or in_features
        self.fc1 = nn.Conv2d(in_features, hidden_features, 1, 1, 0, bias=True)
        self.act = act_layer()
        self.fc2 = nn.Conv2d(hidden_features, out_features, 1, 1, 0, bias=True)
        self.drop = nn.Dropout(drop, inplace=True)

    def forward(self, x):
        x = self.fc1(x); x = self.act(x); x = self.drop(x)
        x = self.fc2(x); x = self.drop(x); return x


class GlobalLocalAttention(nn.Module):
    """Window self-attention (global) combined with a local conv branch."""
    def __init__(self, dim=256, num_heads=16, qkv_bias=False,
                 window_size=8, relative_pos_embedding=True):
        super().__init__()
        self.num_heads = num_heads
        head_dim = dim // self.num_heads
        self.scale = head_dim ** -0.5
        self.ws = window_size
        self.qkv = Conv(dim, 3 * dim, kernel_size=1, bias=qkv_bias)
        self.local1 = ConvBN(dim, dim, kernel_size=3)
        self.local2 = ConvBN(dim, dim, kernel_size=1)
        self.proj = SeparableConvBN(dim, dim, kernel_size=window_size)
        self.attn_x = nn.AvgPool2d(kernel_size=(window_size, 1), stride=1,
                                    padding=(window_size // 2 - 1, 0))
        self.attn_y = nn.AvgPool2d(kernel_size=(1, window_size), stride=1,
                                    padding=(0, window_size // 2 - 1))
        self.relative_pos_embedding = relative_pos_embedding
        if self.relative_pos_embedding:
            self.relative_position_bias_table = nn.Parameter(
                torch.zeros((2 * window_size - 1) * (2 * window_size - 1), num_heads))
            coords_h = torch.arange(self.ws); coords_w = torch.arange(self.ws)
            coords = torch.stack(torch.meshgrid([coords_h, coords_w]))
            coords_flatten = torch.flatten(coords, 1)
            relative_coords = coords_flatten[:, :, None] - coords_flatten[:, None, :]
            relative_coords = relative_coords.permute(1, 2, 0).contiguous()
            relative_coords[:, :, 0] += self.ws - 1; relative_coords[:, :, 1] += self.ws - 1
            relative_coords[:, :, 0] *= 2 * self.ws - 1
            relative_position_index = relative_coords.sum(-1)
            self.register_buffer("relative_position_index", relative_position_index)
            trunc_normal_(self.relative_position_bias_table, std=.02)

    def pad(self, x, ps):
        _, _, H, W = x.size()
        if W % ps != 0: x = F.pad(x, (0, ps - W % ps), mode='reflect')
        if H % ps != 0: x = F.pad(x, (0, 0, 0, ps - H % ps), mode='reflect')
        return x

    def pad_out(self, x):
        return F.pad(x, pad=(0, 1, 0, 1), mode='reflect')

    def forward(self, x):
        B, C, H, W = x.shape
        local = self.local2(x) + self.local1(x)
        x = self.pad(x, self.ws); B, C, Hp, Wp = x.shape
        qkv = self.qkv(x)
        q, k, v = rearrange(qkv, 'b (qkv h d) (hh ws1) (ww ws2) -> qkv (b hh ww) h (ws1 ws2) d',
            h=self.num_heads, d=C // self.num_heads, hh=Hp // self.ws, ww=Wp // self.ws,
            qkv=3, ws1=self.ws, ws2=self.ws)
        dots = (q @ k.transpose(-2, -1)) * self.scale
        if self.relative_pos_embedding:
            relative_position_bias = self.relative_position_bias_table[
                self.relative_position_index.view(-1)].view(self.ws * self.ws, self.ws * self.ws, -1)
            relative_position_bias = relative_position_bias.permute(2, 0, 1).contiguous()
            dots += relative_position_bias.unsqueeze(0)
        attn = dots.softmax(dim=-1); attn = attn @ v
        attn = rearrange(attn, '(b hh ww) h (ws1 ws2) d -> b (h d) (hh ws1) (ww ws2)',
            h=self.num_heads, d=C // self.num_heads, hh=Hp // self.ws, ww=Wp // self.ws,
            ws1=self.ws, ws2=self.ws)
        attn = attn[:, :, :H, :W]
        out = self.attn_x(F.pad(attn, pad=(0, 0, 0, 1), mode='reflect')) + \
              self.attn_y(F.pad(attn, pad=(0, 1, 0, 0), mode='reflect'))
        out = out + local; out = self.pad_out(out); out = self.proj(out)
        out = out[:, :, :H, :W]
        return out

In [ ]:
# Transformer Block (GLTB)
class Block(nn.Module):
    """Global-Local Transformer Block (GLTB): pre-norm attention then MLP, both with residuals."""
    def __init__(self, dim=256, num_heads=16, mlp_ratio=4., qkv_bias=False,
                 drop=0., drop_path=0.,
                 act_layer=nn.ReLU6, norm_layer=nn.BatchNorm2d, window_size=8):
        super().__init__()
        self.norm1 = norm_layer(dim)
        self.attn = GlobalLocalAttention(dim, num_heads=num_heads, qkv_bias=qkv_bias,
                                         window_size=window_size)
        self.drop_path = DropPath(drop_path) if drop_path > 0. else nn.Identity()
        mlp_hidden_dim = int(dim * mlp_ratio)
        self.mlp = Mlp(in_features=dim, hidden_features=mlp_hidden_dim, out_features=dim,
                       act_layer=act_layer, drop=drop)
        self.norm2 = norm_layer(dim)

    def forward(self, x):
        x = x + self.drop_path(self.attn(self.norm1(x)))
        x = x + self.drop_path(self.mlp(self.norm2(x)))
        return x

In [ ]:
# Decoder components: weighted feature fusion (WF), feature refinement head, aux head
class WF(nn.Module):
    """Weighted feature fusion: upsamples low-resolution features and combines them
    with skip-connection features using learned, ReLU-normalized weights."""
    def __init__(self, in_channels=128, decode_channels=128, eps=1e-8):
        super().__init__()
        self.pre_conv = Conv(in_channels, decode_channels, kernel_size=1)
        self.weights = nn.Parameter(torch.ones(2, dtype=torch.float32), requires_grad=True)
        self.eps = eps
        self.post_conv = ConvBNReLU(decode_channels, decode_channels, kernel_size=3)

    def forward(self, x, res):
        x = F.interpolate(x, scale_factor=2, mode='bilinear', align_corners=False)
        weights = nn.ReLU()(self.weights)
        fuse_weights = weights / (torch.sum(weights, dim=0) + self.eps)
        x = fuse_weights[0] * self.pre_conv(res) + fuse_weights[1] * x
        return self.post_conv(x)


class FeatureRefinementHead(nn.Module):
    """Final decoder stage: weighted fusion, spatial and channel attention, residual projection."""
    def __init__(self, in_channels=64, decode_channels=64):
        super().__init__()
        self.pre_conv = Conv(in_channels, decode_channels, kernel_size=1)
        self.weights = nn.Parameter(torch.ones(2, dtype=torch.float32), requires_grad=True)
        self.eps = 1e-8
        self.post_conv = ConvBNReLU(decode_channels, decode_channels, kernel_size=3)
        self.pa = nn.Sequential(
            nn.Conv2d(decode_channels, decode_channels, kernel_size=3, padding=1,
                      groups=decode_channels), nn.Sigmoid())
        self.ca = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            Conv(decode_channels, decode_channels // 16, kernel_size=1), nn.ReLU6(),
            Conv(decode_channels // 16, decode_channels, kernel_size=1), nn.Sigmoid())
        self.shortcut = ConvBN(decode_channels, decode_channels, kernel_size=1)
        self.proj = SeparableConvBN(decode_channels, decode_channels, kernel_size=3)
        self.act = nn.ReLU6()

    def forward(self, x, res):
        x = F.interpolate(x, scale_factor=2, mode='bilinear', align_corners=False)
        weights = nn.ReLU()(self.weights)
        fuse_weights = weights / (torch.sum(weights, dim=0) + self.eps)
        x = fuse_weights[0] * self.pre_conv(res) + fuse_weights[1] * x
        x = self.post_conv(x); shortcut = self.shortcut(x)
        pa = self.pa(x) * x; ca = self.ca(x) * x
        x = pa + ca; x = self.proj(x) + shortcut
        return self.act(x)


class AuxHead(nn.Module):
    """Auxiliary segmentation head used during training for deep supervision."""
    def __init__(self, in_channels=64, num_classes=8):
        super().__init__()
        self.conv = ConvBNReLU(in_channels, in_channels)
        self.drop = nn.Dropout(0.1)
        self.conv_out = Conv(in_channels, num_classes, kernel_size=1)

    def forward(self, x, h, w):
        feat = self.conv(x); feat = self.drop(feat); feat = self.conv_out(feat)
        return F.interpolate(feat, size=(h, w), mode='bilinear', align_corners=False)

## 4. Multimodal architecture

The multimodal model extends UNetFormer with three additions:

1. **Frozen CLIP text encoder** (`ViT-B-32`, `laion2b_s34b_b79k`): converts each caption into a 512-d L2-normalized embedding. No gradients flow into CLIP.
2. **Text-Visual Cross-Attention**: visual features attend to the text embedding via a gated residual. The gate parameter starts at zero, so text influence ramps up gradually during training rather than disrupting the visual features early on.
3. **TextAugmentedDecoder**: same as the UNetFormer decoder, but with a cross-attention module after each Global-Local Transformer Block.

The CLIP text encoder is frozen. The CNN encoder (`swsl_resnet18`), the decoder, and the cross-attention modules are all trained. The backbone is initialized from ImageNet-pretrained weights and fine-tuned during training.

In [ ]:
class CLIPTextEncoder(nn.Module):
    """Frozen CLIP text encoder. Outputs L2-normalized 512-d embeddings."""

    def __init__(self, model_name='ViT-B-32', pretrained='laion2b_s34b_b79k'):
        super().__init__()
        clip_model, _, _ = open_clip.create_model_and_transforms(model_name, pretrained=pretrained)
        self.clip_model = clip_model
        self.tokenizer = open_clip.get_tokenizer(model_name)

        # Drop the visual tower since only the text encoder is used.
        # Saves ~88M frozen params from GPU memory.
        if hasattr(self.clip_model, 'visual'):
            del self.clip_model.visual

        # Freeze all remaining CLIP parameters
        for param in self.parameters():
            param.requires_grad = False

    @torch.no_grad()
    def forward(self, text_list):
        device = next(self.parameters()).device
        tokens = self.tokenizer(text_list).to(device)
        text_features = self.clip_model.encode_text(tokens)
        text_features = text_features / text_features.norm(dim=-1, keepdim=True)
        return text_features


# Verify
clip_enc = CLIPTextEncoder().to(device)
with torch.no_grad():
    emb = clip_enc(['test caption'])
    print(f'CLIP text encoder loaded: output {emb.shape}')
    print(f'Frozen params: {sum(p.numel() for p in clip_enc.parameters()) / 1e6:.1f}M')
del clip_enc

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


open_clip_model.safetensors:   0%|          | 0.00/605M [00:00<?, ?B/s]

CLIP text encoder loaded: output torch.Size([1, 512])
Frozen params: 63.4M


In [ ]:
class TextVisualCrossAttention(nn.Module):
    """Visual features attend to text embedding via gated residual.

    The gate starts at 0, meaning text initially has no influence. As training
    progresses, the gate learns to weight the text contribution.
    """

    def __init__(self, visual_dim=64, text_dim=512, num_heads=4):
        super().__init__()
        self.num_heads = num_heads
        self.head_dim = visual_dim // num_heads
        self.scale = self.head_dim ** -0.5

        # Project text embedding into the visual feature space
        self.text_proj = nn.Sequential(
            nn.Linear(text_dim, visual_dim), nn.ReLU6(),
            nn.Linear(visual_dim, visual_dim),
        )

        self.q_proj = nn.Conv2d(visual_dim, visual_dim, 1)
        self.k_proj = nn.Linear(visual_dim, visual_dim)
        self.v_proj = nn.Linear(visual_dim, visual_dim)
        self.out_proj = nn.Sequential(
            nn.Conv2d(visual_dim, visual_dim, 1), nn.BatchNorm2d(visual_dim),
        )
        # Gated residual — starts at 0, learns to scale text contribution
        self.gate = nn.Parameter(torch.zeros(1))

    def forward(self, visual, text_embed):
        B, C, H, W = visual.shape
        text_feat = self.text_proj(text_embed)
        q = self.q_proj(visual).reshape(B, self.num_heads, self.head_dim, H * W).permute(0, 1, 3, 2)
        k = self.k_proj(text_feat).reshape(B, self.num_heads, 1, self.head_dim)
        v = self.v_proj(text_feat).reshape(B, self.num_heads, 1, self.head_dim)
        attn = (q @ k.transpose(-2, -1)) * self.scale
        attn = attn.softmax(dim=-1)
        out = (attn @ v).permute(0, 1, 3, 2).reshape(B, C, H, W)
        out = self.out_proj(out)
        return visual + self.gate * out

In [ ]:
class TextAugmentedDecoder(nn.Module):
    """UNetFormer decoder + cross-attention after each GLTB."""

    def __init__(self, encoder_channels=(64, 128, 256, 512), decode_channels=64,
                 dropout=0.1, window_size=8, num_classes=7, text_dim=512):
        super().__init__()
        self.pre_conv = ConvBN(encoder_channels[-1], decode_channels, kernel_size=1)
        self.b4 = Block(dim=decode_channels, num_heads=8, window_size=window_size)
        self.p3 = WF(encoder_channels[-2], decode_channels)
        self.b3 = Block(dim=decode_channels, num_heads=8, window_size=window_size)
        self.p2 = WF(encoder_channels[-3], decode_channels)
        self.b2 = Block(dim=decode_channels, num_heads=8, window_size=window_size)
        self.p1 = FeatureRefinementHead(encoder_channels[-4], decode_channels)
        self.up4 = nn.UpsamplingBilinear2d(scale_factor=4)
        self.up3 = nn.UpsamplingBilinear2d(scale_factor=2)
        self.aux_head = AuxHead(decode_channels, num_classes)
        self.segmentation_head = nn.Sequential(
            ConvBNReLU(decode_channels, decode_channels),
            nn.Dropout2d(p=dropout, inplace=True),
            Conv(decode_channels, num_classes, kernel_size=1),
        )

        # Text-visual cross-attention at each decoder stage
        self.text_attn4 = TextVisualCrossAttention(decode_channels, text_dim, num_heads=4)
        self.text_attn3 = TextVisualCrossAttention(decode_channels, text_dim, num_heads=4)
        self.text_attn2 = TextVisualCrossAttention(decode_channels, text_dim, num_heads=4)
        self.text_attn1 = TextVisualCrossAttention(decode_channels, text_dim, num_heads=4)

        self.init_weight()

    def forward(self, res1, res2, res3, res4, h, w, text_embed):
        if self.training:
            x = self.b4(self.pre_conv(res4)); x = self.text_attn4(x, text_embed); h4 = self.up4(x)
            x = self.p3(x, res3); x = self.b3(x); x = self.text_attn3(x, text_embed); h3 = self.up3(x)
            x = self.p2(x, res2); x = self.b2(x); x = self.text_attn2(x, text_embed); h2 = x
            x = self.p1(x, res1); x = self.text_attn1(x, text_embed)
            x = self.segmentation_head(x)
            x = F.interpolate(x, size=(h, w), mode='bilinear', align_corners=False)
            ah = h4 + h3 + h2; ah = self.aux_head(ah, h, w)
            return x, ah
        else:
            x = self.b4(self.pre_conv(res4)); x = self.text_attn4(x, text_embed)
            x = self.p3(x, res3); x = self.b3(x); x = self.text_attn3(x, text_embed)
            x = self.p2(x, res2); x = self.b2(x); x = self.text_attn2(x, text_embed)
            x = self.p1(x, res1); x = self.text_attn1(x, text_embed)
            x = self.segmentation_head(x)
            return F.interpolate(x, size=(h, w), mode='bilinear', align_corners=False)

    def init_weight(self):
        for m in self.children():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, a=1)
                if m.bias is not None:
                    nn.init.constant_(m.bias, 0)

In [ ]:
class UNetFormerCLIP(nn.Module):
    """UNetFormer + frozen CLIP text encoder + cross-attention fusion."""

    def __init__(self, decode_channels=64, dropout=0.1, backbone_name='swsl_resnet18',
                 pretrained=True, window_size=8, num_classes=7,
                 clip_model='ViT-B-32', clip_pretrained='laion2b_s34b_b79k'):
        super().__init__()
        self.backbone = timm.create_model(backbone_name, features_only=True, output_stride=32,
                                          out_indices=(1, 2, 3, 4), pretrained=pretrained)
        encoder_channels = self.backbone.feature_info.channels()
        self.text_encoder = CLIPTextEncoder(clip_model, clip_pretrained)
        self.decoder = TextAugmentedDecoder(encoder_channels, decode_channels, dropout,
                                            window_size, num_classes, text_dim=512)

    def forward(self, x, captions):
        h, w = x.size()[-2:]
        res1, res2, res3, res4 = self.backbone(x)
        text_embed = self.text_encoder(captions)
        if self.training:
            out, ah = self.decoder(res1, res2, res3, res4, h, w, text_embed)
            return out, ah
        else:
            return self.decoder(res1, res2, res3, res4, h, w, text_embed)

In [ ]:
# Quick sanity check: instantiate, count parameters, forward pass in both modes
test_model = UNetFormerCLIP(num_classes=NUM_CLASSES).to(device)

total_params     = sum(p.numel() for p in test_model.parameters())
trainable_params = sum(p.numel() for p in test_model.parameters() if p.requires_grad)
frozen_params    = total_params - trainable_params
print(f'Total parameters    : {total_params / 1e6:.2f}M')
print(f'Trainable parameters: {trainable_params / 1e6:.2f}M')
print(f'Frozen parameters   : {frozen_params / 1e6:.2f}M (CLIP text encoder)')

with torch.no_grad():
    sample_imgs, _, sample_caps = next(iter(train_loader))
    sample_imgs = sample_imgs[:2].to(device)
    sample_caps = list(sample_caps[:2])

    test_model.eval()
    out_eval = test_model(sample_imgs, sample_caps)
    print(f'\nEval  mode: input {sample_imgs.shape} -> output {out_eval.shape}')

    test_model.train()
    out_main, out_aux = test_model(sample_imgs, sample_caps)
    print(f'Train mode: main {out_main.shape}, aux {out_aux.shape}')

del test_model
torch.cuda.empty_cache()

/usr/local/lib/python3.12/dist-packages/timm/models/_factory.py:138: UserWarning: Mapping deprecated model name swsl_resnet18 to current resnet18.fb_swsl_ig1b_ft_in1k.
  model = create_fn(


model.safetensors:   0%|          | 0.00/46.8M [00:00<?, ?B/s]

Total parameters    : 75.37M
Trainable parameters: 11.94M
Frozen parameters   : 63.43M (CLIP text encoder)


/usr/local/lib/python3.12/dist-packages/torch/functional.py:505: UserWarning: torch.meshgrid: in an upcoming release, it will be required to pass the indexing argument. (Triggered internally at /pytorch/aten/src/ATen/native/TensorShape.cpp:4381.)
  return _VF.meshgrid(tensors, **kwargs)  # type: ignore[attr-defined]



Eval  mode: input torch.Size([2, 3, 256, 256]) -> output torch.Size([2, 7, 256, 256])
Train mode: main torch.Size([2, 7, 256, 256]), aux torch.Size([2, 7, 256, 256])


## 5. Training infrastructure

Training and evaluation functions for the multimodal model. Same hyperparameters as the UNetFormer baseline (30 epochs, AdamW lr=6e-4, weight decay 0.01, CosineAnnealingLR, weighted CE loss + 0.4 × auxiliary head loss).

The training function accepts an optional `kl_weight` argument. When non-zero, an additional KL-divergence term is added between the predicted class distribution and the image-level class distribution from the CSV. This is used in the auxiliary regularizer ablation.

Each training run is logged to Weights & Biases under the project `di725_project`.

In [ ]:
# Training hyperparameters (same as UNetFormer baseline)
NUM_EPOCHS      = 30
LR              = 6e-4
WEIGHT_DECAY    = 0.01
AUX_LOSS_WEIGHT = 0.4

In [ ]:
# Pixel-level IoU helpers — accumulate intersection/union across the full dataset
def update_confusion(intersection, union, pred, target, num_classes=NUM_CLASSES):
    """Accumulate per-class intersection and union counts in-place.
    pred, target: tensors of shape (B, H, W) with class indices.
    Computation stays on GPU; only scalars transfer to CPU via .item()."""
    for c in range(num_classes):
        pred_c   = (pred == c)
        target_c = (target == c)
        intersection[c] += (pred_c & target_c).sum().item()
        union[c]        += (pred_c | target_c).sum().item()


def finalize_iou(intersection, union, num_classes=NUM_CLASSES):
    """Convert accumulated counts into per-class IoU and mIoU.
    Classes absent from both pred and target (union=0) are NaN and excluded from mIoU."""
    class_ious = []
    for c in range(num_classes):
        if union[c] == 0:
            class_ious.append(float('nan'))
        else:
            class_ious.append(intersection[c] / union[c])
    valid = [x for x in class_ious if not np.isnan(x)]
    miou = np.mean(valid) if valid else 0.0
    return class_ious, miou

In [ ]:
def train_one_epoch(model, loader, optimizer, criterion, device, kl_weight=0.0):
    """One training epoch.

    If kl_weight > 0, adds KL-divergence term between predicted class
    distribution (spatial mean of softmax) and image-level class distribution
    computed from the mask. Acts as a train-time auxiliary regularizer.
    """
    model.train()
    total_loss = 0
    for imgs, masks, captions in loader:
        imgs, masks = imgs.to(device), masks.to(device)
        logits_main, logits_aux = model(imgs, list(captions))

        loss = criterion(logits_main, masks) + AUX_LOSS_WEIGHT * criterion(logits_aux, masks)

        if kl_weight > 0:
            pred_dist = F.softmax(logits_main, dim=1).mean(dim=[2, 3])
            gt_dist = torch.zeros_like(pred_dist)
            for c in range(NUM_CLASSES):
                gt_dist[:, c] = (masks == c).float().mean(dim=[1, 2])
            kl_loss = F.kl_div(
                torch.log(pred_dist + 1e-8), gt_dist + 1e-8,
                reduction='batchmean',
            )
            loss = loss + kl_weight * kl_loss

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    return total_loss / len(loader)


@torch.no_grad()
def evaluate(model, loader, criterion, device, num_classes=NUM_CLASSES):
    """Returns: per-class pixel IoU list, pixel-level mIoU, overall accuracy, average loss."""
    model.eval()
    intersection = np.zeros(num_classes, dtype=np.int64)
    union        = np.zeros(num_classes, dtype=np.int64)
    total_correct = 0
    total_pixels  = 0
    total_loss    = 0
    for imgs, masks, captions in loader:
        imgs, masks = imgs.to(device), masks.to(device)
        logits = model(imgs, list(captions))
        total_loss += criterion(logits, masks).item()
        preds = logits.argmax(dim=1)
        update_confusion(intersection, union, preds, masks, num_classes)
        total_correct += (preds == masks).sum().item()
        total_pixels  += masks.numel()
    class_ious, miou = finalize_iou(intersection, union, num_classes)
    oa = total_correct / total_pixels
    return class_ious, miou, oa, total_loss / len(loader)

In [ ]:
def train_multimodal(caption_col, num_epochs=NUM_EPOCHS, lr=LR, kl_weight=0.0,
                     run_suffix=None):
    """Train UNetFormerCLIP for a given caption column.

    Args:
        caption_col: which caption column from CAPTION_COLS to use as text input.
        num_epochs: number of epochs to train.
        lr: initial learning rate.
        kl_weight: weight on the KL-divergence auxiliary loss (0 = disabled).
        run_suffix: optional string appended to the run name (e.g. '+kl').

    Returns:
        history dict, best_miou, save_path.
    """
    # Build identifiers
    base_name = caption_col.replace('-', '_').replace('/', '_')
    run_name = f'multimodal_{base_name}'
    if run_suffix:
        run_name += f'_{run_suffix}'
    save_path = CHECKPOINTS_DIR / f'{run_name}_best.pth'

    # Loaders for this caption variant
    train_loader, val_loader, _ = make_loaders(caption_col)

    # Model
    model = UNetFormerCLIP(num_classes=NUM_CLASSES).to(device)

    # Optimizer over only the trainable params (skips frozen CLIP)
    trainable = filter(lambda p: p.requires_grad, model.parameters())
    optimizer = AdamW(trainable, lr=lr, weight_decay=WEIGHT_DECAY)
    scheduler = CosineAnnealingLR(optimizer, T_max=num_epochs, eta_min=1e-6)
    criterion = nn.CrossEntropyLoss(weight=class_weights_tensor)

    # wandb run
    run = wandb.init(
        project=WANDB_PROJECT,
        name=run_name,
        config={
            'model':           'UNetFormerCLIP',
            'caption_col':     caption_col,
            'kl_weight':       kl_weight,
            'num_epochs':      num_epochs,
            'lr':              lr,
            'weight_decay':    WEIGHT_DECAY,
            'batch_size':      BATCH_SIZE,
            'aux_loss_weight': AUX_LOSS_WEIGHT,
            'seed':            SEED,
        },
        reinit=True,
    )

    history = {'train_loss': [], 'val_loss': [], 'val_miou': [], 'val_oa': []}
    best_miou = 0.0

    print(f'Training {run_name} for {num_epochs} epochs...')
    print(f"{'Epoch':>5} {'T_Loss':>8} {'V_Loss':>8} {'V_mIoU':>8} {'V_OA':>8} {'Time':>7} {'':>6}")
    print('-' * 55)

    for epoch in range(num_epochs):
        t0 = time.time()
        train_loss = train_one_epoch(model, train_loader, optimizer, criterion, device,
                                     kl_weight=kl_weight)
        val_class_ious, val_miou, val_oa, val_loss = evaluate(model, val_loader, criterion, device)
        scheduler.step()

        history['train_loss'].append(train_loss)
        history['val_loss'].append(val_loss)
        history['val_miou'].append(val_miou)
        history['val_oa'].append(val_oa)

        log_dict = {
            'epoch':      epoch + 1,
            'train_loss': train_loss,
            'val_loss':   val_loss,
            'val_miou':   val_miou,
            'val_oa':     val_oa,
            'lr':         scheduler.get_last_lr()[0],
        }
        for c, name in enumerate(CLASS_NAMES):
            if not np.isnan(val_class_ious[c]):
                log_dict[f'val_iou/{name}'] = val_class_ious[c]
        wandb.log(log_dict)

        status = ''
        if val_miou > best_miou:
            best_miou = val_miou
            torch.save(model.state_dict(), str(save_path))
            status = 'BEST'

        elapsed = time.time() - t0
        print(f'{epoch+1:>5} {train_loss:>8.4f} {val_loss:>8.4f} {val_miou:>8.4f} {val_oa:>8.4f} {elapsed:>6.1f}s {status:>6}')

    wandb.summary['best_val_miou'] = best_miou
    wandb.finish()

    print(f'\nBest validation mIoU: {best_miou:.4f}')
    return history, best_miou, save_path


def save_history(history, name):
    """Persist training history to disk."""
    path = RESULTS_DIR / f'{name}_history.json'
    with open(path, 'w') as f:
        json.dump(history, f)
    print(f'Saved history to {path}')

## 6. Caption-variant ablation

Train UNetFormerCLIP separately for each of the 5 caption variants and compare them against the vision-only UNetFormer baseline. The goal: identify which caption variant contributes most to segmentation performance.

The 5 variants:

- **Hybrid** (`hybrid_gemma3-4b`, `hybrid_qwen3-vl-8b`)
- **Text-only** (`text_qwen3-4b`)
- **Vision-only** (`vision_gemma3-4b`, `vision_qwen3-vl-8b`)

Each run uses the same hyperparameters as the UNetFormer baseline. All runs are tracked in Weights & Biases under the `di725_project` project.

The vision-only baseline (UNetFormer without text) from `03_baseline_models.ipynb` provides the reference point.

In [ ]:
# Run 1/5: hybrid_gemma3-4b
hist_hyb_gemma, miou_hyb_gemma, ckpt_hyb_gemma = train_multimodal('hybrid_gemma3-4b')
save_history(hist_hyb_gemma, 'multimodal_hybrid_gemma3_4b')

wandb: WARNING Using a boolean value for 'reinit' is deprecated. Use 'return_previous' or 'finish_previous' instead.


Training multimodal_hybrid_gemma3_4b for 30 epochs...
Epoch   T_Loss   V_Loss   V_mIoU     V_OA    Time       
-------------------------------------------------------
    1   1.0993   0.5501   0.4191   0.7040   64.1s   BEST
    2   0.7491   0.4728   0.4782   0.7276   56.6s   BEST
    3   0.6270   0.3927   0.5457   0.7576   56.9s   BEST
    4   0.6281   0.3998   0.5682   0.7811   57.3s   BEST
    5   0.5550   0.4174   0.5514   0.7635   56.7s       
    6   0.5371   0.3718   0.5394   0.7668   57.1s       
    7   0.5110   0.3698   0.5482   0.7910   57.2s       
    8   0.5041   0.2882   0.6094   0.8270   56.7s   BEST
    9   0.4508   0.2836   0.6049   0.8152   56.4s       
   10   0.4474   0.2790   0.6156   0.8374   57.2s   BEST
   11   0.4247   0.3025   0.5768   0.8037   56.4s       
   12   0.4073   0.2812   0.6167   0.8326   57.0s   BEST
   13   0.3972   0.2628   0.6420   0.8437   57.3s   BEST
   14   0.3831   0.2713   0.6441   0.8530   57.4s   BEST
   15   0.3626   0.2503   0.6520   

epoch,▁▁▁▂▂▂▂▃▃▃▃▄▄▄▄▅▅▅▅▆▆▆▆▇▇▇▇███
lr,█████▇▇▇▇▆▆▆▅▅▅▄▄▃▃▃▂▂▂▂▁▁▁▁▁▁
train_loss,█▅▄▄▄▃▃▃▃▃▂▂▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁
val_iou/Barren,▃▁▂▅▄▄▄▅▅▆▃▆▅▇▇▇▆▆▅▆▅▆▇▇█▇▇██▇
val_iou/Built-up,▃▁▅▆▅▃▅▅▆▄▅▅▇▅▆█▇▇█▇▆▇▇▇▇▇██▇█
val_iou/Crop,▄▁▅▄▄▄▅▆▆▇▆▇▇▇▇▇▇▇███▇████████
val_iou/Grass,▁▁▁▂▃▂▃▅▄▆▄▅▆▇▇▇▆▇▇▇▇▇█▇██████
val_iou/Shrub,▂▂▁▁▂▁▃▃▂▄▄▄▆▄▄▄▄▆▆█▅▆▇▇▇▇▆██▇
val_iou/Tree,▁▅▄▅▂▅▅▆▆▆▆▇▇▇▇▇▇▇███▇████████
val_iou/Water,▁▆▇▇▇▇▆██▇▇▇█████▇████████████
+3,...



Best validation mIoU: 0.6963
Saved history to /content/drive/MyDrive/DI725_Project/results/multimodal_hybrid_gemma3_4b_history.json


In [ ]:
# Run 2/5: hybrid_qwen3-vl-8b
hist_hyb_qwen, miou_hyb_qwen, ckpt_hyb_qwen = train_multimodal('hybrid_qwen3-vl-8b')
save_history(hist_hyb_qwen, 'multimodal_hybrid_qwen3_vl_8b')

/usr/local/lib/python3.12/dist-packages/timm/models/_factory.py:138: UserWarning: Mapping deprecated model name swsl_resnet18 to current resnet18.fb_swsl_ig1b_ft_in1k.
  model = create_fn(


Training multimodal_hybrid_qwen3_vl_8b for 30 epochs...
Epoch   T_Loss   V_Loss   V_mIoU     V_OA    Time       
-------------------------------------------------------
    1   1.1032   0.4712   0.5141   0.7461   63.2s   BEST
    2   0.7126   0.4046   0.4962   0.7598   55.8s       
    3   0.6376   0.3534   0.5491   0.7857   57.5s   BEST
    4   0.5654   0.4131   0.5834   0.8087   57.7s   BEST
    5   0.5391   0.4308   0.5599   0.7990   56.7s       
    6   0.5140   0.3263   0.5853   0.8218   57.2s   BEST
    7   0.4860   0.3015   0.5770   0.8042   56.6s       
    8   0.4607   0.3301   0.5848   0.8009   56.2s       
    9   0.4461   0.4646   0.5927   0.8024   57.0s   BEST
   10   0.4248   0.2890   0.6012   0.8200   57.5s   BEST
   11   0.4111   0.2776   0.6144   0.8216   56.7s   BEST
   12   0.4003   0.2779   0.6395   0.8513   57.3s   BEST
   13   0.3868   0.2533   0.6542   0.8572   56.8s   BEST
   14   0.3627   0.2560   0.6179   0.8395   56.3s       
   15   0.3649   0.2525   0.6449 

epoch,▁▁▁▂▂▂▂▃▃▃▃▄▄▄▄▅▅▅▅▆▆▆▆▇▇▇▇███
lr,█████▇▇▇▇▆▆▆▅▅▅▄▄▃▃▃▂▂▂▂▁▁▁▁▁▁
train_loss,█▅▄▄▃▃▃▃▃▂▂▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁
val_iou/Barren,▁▁▃▄▄▅▂▅▅▃▃▅▆▄▄▇▆█▇▅▇▆▆███▇▇█▇
val_iou/Built-up,▁▃▂▅▄▅▅▃▇▃▆▅▆▃█▇▆▅▇▆▇▇▇█▇▇█▇▇▇
val_iou/Crop,▁▄▄▄▁▅▅▂▁▅▅▆▇▆▇▇▆█▇▇██████████
val_iou/Grass,▁▂▃▄▃▅▄▃▃▄▄▆▇▆▆▇▆█▆▆▇▇▇███████
val_iou/Shrub,▁▂▂▃▅▂▂▄▄▄▅▅▅▄▄▆▆▅▄▅▅▆▇▆▇█▇▇█▇
val_iou/Tree,▂▁▂▄▅▄▅▄▄▆▆▇▇▇▇▇▇▇▇▇▇███▇█████
val_iou/Water,▆▁▅▆▄▄▅▇▅█▇▇███▇▆██▆▇█████████
+3,...



Best validation mIoU: 0.6949
Saved history to /content/drive/MyDrive/DI725_Project/results/multimodal_hybrid_qwen3_vl_8b_history.json


In [ ]:
# Run 3/5: text_qwen3-4b
hist_text_qwen, miou_text_qwen, ckpt_text_qwen = train_multimodal('text_qwen3-4b')
save_history(hist_text_qwen, 'multimodal_text_qwen3_4b')

Training multimodal_text_qwen3_4b for 30 epochs...
Epoch   T_Loss   V_Loss   V_mIoU     V_OA    Time       
-------------------------------------------------------
    1   1.1499   0.5395   0.4724   0.7408   62.9s   BEST
    2   0.7374   0.4258   0.4918   0.7303   56.7s   BEST
    3   0.6909   0.4684   0.5073   0.7711   58.1s   BEST
    4   0.6024   0.4023   0.5921   0.8066   56.8s   BEST
    5   0.5728   0.3444   0.5853   0.8170   56.1s       
    6   0.5528   0.3370   0.5531   0.7894   56.4s       
    7   0.5176   0.3473   0.5744   0.8039   55.8s       
    8   0.5061   0.3164   0.5791   0.8138   56.5s       
    9   0.4719   0.3209   0.6148   0.8243   57.3s   BEST
   10   0.4588   0.4817   0.4998   0.7852   56.7s       
   11   0.4457   0.3064   0.5940   0.8198   56.7s       
   12   0.4163   0.2903   0.6151   0.8312   57.0s   BEST
   13   0.3989   0.2595   0.6143   0.8268   56.2s       
   14   0.3925   0.2766   0.6178   0.8342   56.4s   BEST
   15   0.3745   0.2586   0.6583   0.8

epoch,▁▁▁▂▂▂▂▃▃▃▃▄▄▄▄▅▅▅▅▆▆▆▆▇▇▇▇███
lr,█████▇▇▇▇▆▆▆▅▅▅▄▄▃▃▃▂▂▂▂▁▁▁▁▁▁
train_loss,█▅▄▄▃▃▃▃▃▃▂▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁
val_iou/Barren,▂▁▂▄▅▃▂▃▅▄▄▄▃▃▆▆▆▆▇▇▆███▇▇██▇▇
val_iou/Built-up,▅▁▅█▆▅▅▅▇▆▅▆▇▇█▇█▆▆█▇▇▇█▇█████
val_iou/Crop,▁▃▃▄▅▅▄▅▅▅▆▅▆▇▇▇▆▇█▇▇█████████
val_iou/Grass,▂▁▄▅▅▄▄▅▅▅▆▆▅▅▇▇▆▇▇█▇████▇████
val_iou/Shrub,▁▂▂▂▃▁▅▃▃▄▄▄▄▆▅▃▆▅▅▇▄▇▇█████▇█
val_iou/Tree,▄▅▄▆▆▅▆▆▆▃▅▇▇▇▇▇▁▇▇█▇█████████
val_iou/Water,▃▇▅▇▇▆▇▇▇▂▇██▇██▁█████████████
+3,...



Best validation mIoU: 0.6982
Saved history to /content/drive/MyDrive/DI725_Project/results/multimodal_text_qwen3_4b_history.json


In [ ]:
# Run 4/5: vision_gemma3-4b
hist_vis_gemma, miou_vis_gemma, ckpt_vis_gemma = train_multimodal('vision_gemma3-4b')
save_history(hist_vis_gemma, 'multimodal_vision_gemma3_4b')

Training multimodal_vision_gemma3_4b for 30 epochs...
Epoch   T_Loss   V_Loss   V_mIoU     V_OA    Time       
-------------------------------------------------------
    1   1.3149   0.6822   0.4014   0.6723   63.8s   BEST
    2   1.0124   0.5623   0.4416   0.6918   57.3s   BEST
    3   0.9005   0.5489   0.4800   0.6962   58.3s   BEST
    4   0.8122   0.5147   0.4635   0.7109   58.6s       
    5   0.7679   0.4329   0.5166   0.7572   58.9s   BEST
    6   0.7454   0.5552   0.4761   0.7295   57.4s       
    7   0.7336   0.5802   0.4389   0.6623   57.9s       
    8   0.6753   0.5376   0.5344   0.7531   58.3s   BEST
    9   0.6524   0.4754   0.4873   0.7245   57.9s       
   10   0.6211   0.4202   0.5441   0.7619   57.6s   BEST
   11   0.5943   0.4858   0.5388   0.7277   57.1s       
   12   0.5719   0.3629   0.5736   0.7979   58.1s   BEST
   13   0.5487   0.3821   0.5616   0.8044   57.4s       
   14   0.5375   0.4056   0.5533   0.7869   58.1s       
   15   0.5178   0.3897   0.6026   

epoch,▁▁▁▂▂▂▂▃▃▃▃▄▄▄▄▅▅▅▅▆▆▆▆▇▇▇▇███
lr,█████▇▇▇▇▆▆▆▅▅▅▄▄▃▃▃▂▂▂▂▁▁▁▁▁▁
train_loss,█▆▅▄▄▄▄▃▃▃▃▃▂▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁
val_iou/Barren,▁▂▁▁▅▃▃▅▃▆▆▆▅▄▆▅▅▅▇▅▇▇▇█▇█▇█▇█
val_iou/Built-up,▁▃▅▃▅▇▅▆▇▅▆▅▆▅▆█▆▇▇▆▇▇▇▇▆▇▇▇▇▇
val_iou/Crop,▄▄▄▅▆▄▁▄▄▆▆▇▆▆▆▅▆▆▇▇▇█▇███████
val_iou/Grass,▁▂▁▂▅▃▁▄▃▄▃▆▆▅▆▅▆▅▆▆▇▇▇▇██████
val_iou/Shrub,▁▁▂▃▂▃▂▄▄▂▁▃▅▆▆▃▄▅▄▇▇▇▇▆███▇██
val_iou/Tree,▂▃▄▄▃▃▁▃▃▅▃▆▆▅▆▄▅▅▆▇▇▇█▇██████
val_iou/Water,▂▃▅▄▅▁▃▆▂▇▇▇▅▆█▇▇█▇█▇▇████████
+3,...



Best validation mIoU: 0.6494
Saved history to /content/drive/MyDrive/DI725_Project/results/multimodal_vision_gemma3_4b_history.json


In [ ]:
# Run 5/5: vision_qwen3-vl-8b
hist_vis_qwen, miou_vis_qwen, ckpt_vis_qwen = train_multimodal('vision_qwen3-vl-8b')
save_history(hist_vis_qwen, 'multimodal_vision_qwen3_vl_8b')

Training multimodal_vision_qwen3_vl_8b for 30 epochs...
Epoch   T_Loss   V_Loss   V_mIoU     V_OA    Time       
-------------------------------------------------------
    1   1.3013   1.1141   0.3282   0.6025   60.8s   BEST
    2   1.0197   0.6139   0.4622   0.6790   58.6s   BEST
    3   0.9091   0.6064   0.4731   0.7454   57.6s   BEST
    4   0.8416   0.6219   0.4694   0.6492   57.4s       
    5   0.7675   0.5971   0.4415   0.7035   58.4s       
    6   0.7474   0.5303   0.4537   0.6314   56.9s       
    7   0.7067   0.4631   0.4886   0.7052   58.8s   BEST
    8   0.6805   0.4192   0.5314   0.7424   59.0s   BEST
    9   0.6445   0.4068   0.5358   0.7347   58.8s   BEST
   10   0.6178   0.4208   0.5114   0.7638   57.4s       
   11   0.5792   0.4186   0.5558   0.7764   58.1s   BEST
   12   0.5770   0.4189   0.5609   0.7714   57.9s   BEST
   13   0.5386   0.3859   0.5684   0.7902   58.0s   BEST
   14   0.5511   0.4697   0.5572   0.7649   57.3s       
   15   0.5055   0.3835   0.5775 

epoch,▁▁▁▂▂▂▂▃▃▃▃▄▄▄▄▅▅▅▅▆▆▆▆▇▇▇▇███
lr,█████▇▇▇▇▆▆▆▅▅▅▄▄▃▃▃▂▂▂▂▁▁▁▁▁▁
train_loss,█▆▅▅▄▄▄▃▃▃▃▃▂▃▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁
val_iou/Barren,▂▂▃▃▃▁▄▄▄▄▆▅▄▄▅▇▄▆▆▇▆▆▇█▇▆▇▇▇█
val_iou/Built-up,▅▅▁▆▁▅▄▅▆▇▄▆▅▇▆▇▅▆▇█▇▇▇▇█▇██▇▇
val_iou/Crop,▁▁▅▄▃▅▅▆▄▅▅▆▇▆▆▇▇▇▇█▇█████████
val_iou/Grass,▃▃▅▁▅▃▅▅▅▆▆▅▆▄▆▆▆▇▇▇▇▇███▇████
val_iou/Shrub,▁▄▅▂▄▂▃▃▃▅▄▄▅▄▅▄▅▇▇▇▆▇▇▇▇▇▇█▇█
val_iou/Tree,▁▆▆▅▃▄▄▅▆▆▇▇▇▆▇▇▇█████████████
val_iou/Water,▁▆▅▇▅▆▇▇█▅████▇████▇▇█████████
+3,...



Best validation mIoU: 0.6555
Saved history to /content/drive/MyDrive/DI725_Project/results/multimodal_vision_qwen3_vl_8b_history.json


In [ ]:
# Summary of caption-variant ablation (loaded from saved history files)
caption_history_files = {
    'hybrid_gemma3-4b':    'multimodal_hybrid_gemma3_4b_history.json',
    'hybrid_qwen3-vl-8b':  'multimodal_hybrid_qwen3_vl_8b_history.json',
    'text_qwen3-4b':       'multimodal_text_qwen3_4b_history.json',
    'vision_gemma3-4b':    'multimodal_vision_gemma3_4b_history.json',
    'vision_qwen3-vl-8b':  'multimodal_vision_qwen3_vl_8b_history.json',
}

caption_results = {}
for cap, fname in caption_history_files.items():
    path = RESULTS_DIR / fname
    if path.exists():
        with open(path) as f:
            history = json.load(f)
        caption_results[cap] = max(history['val_miou'])

print(f"{'Caption variant':<25} {'Best val mIoU':>15}")
print('-' * 42)
for cap, miou in sorted(caption_results.items(), key=lambda x: -x[1]):
    print(f'{cap:<25} {miou:>15.4f}')

best_caption = max(caption_results, key=caption_results.get)
print(f'\nBest caption variant: {best_caption} (val mIoU = {caption_results[best_caption]:.4f})')

Caption variant             Best val mIoU
------------------------------------------
text_qwen3-4b                      0.6982
hybrid_gemma3-4b                   0.6963
hybrid_qwen3-vl-8b                 0.6949
vision_qwen3-vl-8b                 0.6555
vision_gemma3-4b                   0.6494

Best caption variant: text_qwen3-4b (val mIoU = 0.6982)


### Test set evaluation

Evaluate each of the 5 multimodal checkpoints on the held-out test set and compare against the vision-only UNetFormer baseline from `03_baseline_models.ipynb`.

In [ ]:
# Load the vision-only baseline result for the comparison table
with open(RESULTS_DIR / 'baselines_results.json') as f:
    baseline_data = json.load(f)

unet_baseline_test_miou  = baseline_data['unetformer']['test_miou']
unet_baseline_test_oa    = baseline_data['unetformer']['test_oa']
unet_baseline_class_ious = baseline_data['unetformer']['class_ious']

print(f'Vision-only UNetFormer baseline:')
print(f'  test mIoU: {unet_baseline_test_miou:.4f}')
print(f'  test OA  : {unet_baseline_test_oa:.4f}')

Vision-only UNetFormer baseline:
  test mIoU: 0.6488
  test OA  : 0.8621


In [ ]:
def evaluate_checkpoint(caption_col, ckpt_path):
    """Reload a trained UNetFormerCLIP checkpoint and evaluate on the test set."""
    _, _, test_loader = make_loaders(caption_col)

    model = UNetFormerCLIP(num_classes=NUM_CLASSES).to(device)
    model.load_state_dict(torch.load(str(ckpt_path)))
    criterion = nn.CrossEntropyLoss(weight=class_weights_tensor)

    class_ious, miou, oa, loss = evaluate(model, test_loader, criterion, device)

    # Free memory before next eval
    del model
    torch.cuda.empty_cache()

    return class_ious, miou, oa, loss

In [ ]:
# Evaluate each caption-variant checkpoint on the test set
multimodal_runs = [
    ('hybrid_gemma3-4b',   CHECKPOINTS_DIR / 'multimodal_hybrid_gemma3_4b_best.pth'),
    ('hybrid_qwen3-vl-8b', CHECKPOINTS_DIR / 'multimodal_hybrid_qwen3_vl_8b_best.pth'),
    ('text_qwen3-4b',      CHECKPOINTS_DIR / 'multimodal_text_qwen3_4b_best.pth'),
    ('vision_gemma3-4b',   CHECKPOINTS_DIR / 'multimodal_vision_gemma3_4b_best.pth'),
    ('vision_qwen3-vl-8b', CHECKPOINTS_DIR / 'multimodal_vision_qwen3_vl_8b_best.pth'),
]

multimodal_test_results = {}
for caption_col, ckpt_path in multimodal_runs:
    print(f'Evaluating {caption_col}...')
    class_ious, miou, oa, loss = evaluate_checkpoint(caption_col, ckpt_path)
    multimodal_test_results[caption_col] = {
        'class_ious': class_ious,
        'miou':       miou,
        'oa':         oa,
        'loss':       loss,
    }
    print(f'  test mIoU: {miou:.4f}, OA: {oa:.4f}')

Evaluating hybrid_gemma3-4b...
  test mIoU: 0.6930, OA: 0.8863
Evaluating hybrid_qwen3-vl-8b...
  test mIoU: 0.6906, OA: 0.8847
Evaluating text_qwen3-4b...
  test mIoU: 0.6970, OA: 0.8894
Evaluating vision_gemma3-4b...
  test mIoU: 0.6393, OA: 0.8568
Evaluating vision_qwen3-vl-8b...
  test mIoU: 0.6519, OA: 0.8609


### Caption-variant comparison vs baseline

Per-class IoU and mIoU comparison across the 5 caption variants and the vision-only UNetFormer baseline. Identifies which caption variant contributes most to segmentation performance.

In [ ]:
# Build comparison table: per-class IoU, mIoU, OA across baseline + 5 original multimodal variants

# Short labels for column headers
short_labels = {
    'hybrid_gemma3-4b':    'hyb_gemma',
    'hybrid_qwen3-vl-8b':  'hyb_qwen',
    'text_qwen3-4b':       'text_qwen',
    'vision_gemma3-4b':    'vis_gemma',
    'vision_qwen3-vl-8b':  'vis_qwen',
}

ORIGINAL_CAPTIONS = list(short_labels.keys())

# Build column header
column_labels = ['vision-only'] + [short_labels[cap] for cap in ORIGINAL_CAPTIONS]
header = f"{'Class':<11}" + ''.join(f'{lbl:>14}' for lbl in column_labels)
print(header)
print('-' * len(header))

# Per-class rows
for i, name in enumerate(CLASS_NAMES):
    row = f'{name:<11}'
    # baseline
    bv = unet_baseline_class_ious[i]
    row += f'{bv:>14.4f}' if bv is not None else f'{"N/A":>14}'
    # multimodal variants
    for caption_col in ORIGINAL_CAPTIONS:
        v = multimodal_test_results[caption_col]['class_ious'][i]
        row += f'{v:>14.4f}' if not np.isnan(v) else f'{"N/A":>14}'
    print(row)

print('-' * len(header))

# mIoU row
miou_row = f'{"mIoU":<11}{unet_baseline_test_miou:>14.4f}'
for caption_col in ORIGINAL_CAPTIONS:
    miou_row += f'{multimodal_test_results[caption_col]["miou"]:>14.4f}'
print(miou_row)

# OA row
oa_row = f'{"OA":<11}{unet_baseline_test_oa:>14.4f}'
for caption_col in ORIGINAL_CAPTIONS:
    oa_row += f'{multimodal_test_results[caption_col]["oa"]:>14.4f}'
print(oa_row)

Class         vision-only     hyb_gemma      hyb_qwen     text_qwen     vis_gemma      vis_qwen
-----------------------------------------------------------------------------------------------
Tree               0.8608        0.8772        0.8738        0.8758        0.8641        0.8659
Shrub              0.2476        0.3591        0.3410        0.3261        0.2665        0.2948
Grass              0.7631        0.7935        0.7963        0.8057        0.7530        0.7563
Crop               0.7416        0.8089        0.8043        0.8159        0.7315        0.7413
Built-up           0.5631        0.5777        0.5746        0.5893        0.5167        0.5621
Barren             0.4869        0.5582        0.5530        0.5665        0.4627        0.4556
Water              0.8784        0.8763        0.8909        0.8994        0.8808        0.8875
-----------------------------------------------------------------------------------------------
mIoU               0.6488        0.6930 

In [ ]:
# Δ vs baseline summary — original 5 caption variants only
ORIGINAL_CAPTIONS = [
    'hybrid_gemma3-4b',
    'hybrid_qwen3-vl-8b',
    'text_qwen3-4b',
    'vision_gemma3-4b',
    'vision_qwen3-vl-8b',
]

print(f"\n{'Caption variant':<25} {'Test mIoU':>10} {'Δ vs baseline':>16} {'Relative %':>12}")
print('-' * 65)
print(f'{"vision-only (baseline)":<25} {unet_baseline_test_miou:>10.4f} {"--":>16} {"--":>12}')

for caption_col in ORIGINAL_CAPTIONS:
    miou = multimodal_test_results[caption_col]['miou']
    delta = miou - unet_baseline_test_miou
    rel = delta / unet_baseline_test_miou * 100
    print(f'{caption_col:<25} {miou:>10.4f} {delta:>+16.4f} {rel:>+11.1f}%')

# Best by test mIoU (only among original variants)
best_caption_test = max(ORIGINAL_CAPTIONS, key=lambda k: multimodal_test_results[k]['miou'])
best_test_miou = multimodal_test_results[best_caption_test]['miou']
print(f'\nBest caption variant on test set: {best_caption_test} (mIoU = {best_test_miou:.4f})')


Caption variant            Test mIoU    Δ vs baseline   Relative %
-----------------------------------------------------------------
vision-only (baseline)        0.6488               --           --
hybrid_gemma3-4b              0.6930          +0.0442        +6.8%
hybrid_qwen3-vl-8b            0.6906          +0.0418        +6.4%
text_qwen3-4b                 0.6970          +0.0482        +7.4%
vision_gemma3-4b              0.6393          -0.0095        -1.5%
vision_qwen3-vl-8b            0.6519          +0.0031        +0.5%

Best caption variant on test set: text_qwen3-4b (mIoU = 0.6970)


**Observations**

Text-rich captions (`text_qwen3-4b`, `hybrid_gemma3-4b`, `hybrid_qwen3-vl-8b`) all improve segmentation by +6.4% to +7.4% relative mIoU. Vision-only captions, in contrast, contribute negligibly (±1.5%). The result aligns with the caption accuracy analysis from `01_data_exploration.ipynb`: vision-only captions miss dominant classes such as Grass and Shrub at very high rates (>94%), so the text branch receives misleading content. Caption quality, not multimodal architecture alone, drives the improvement.

`text_qwen3-4b` has the highest validation mIoU (0.6982) and is therefore selected as the base configuration for the KL-divergence ablation in Section 8. On the held-out test set it confirms this with the highest test mIoU (0.6970).

In [ ]:
# Detailed per-class comparison: vision-only baseline vs best multimodal variant
best_results = multimodal_test_results[best_caption_test]

print(f"Per-class comparison: vision-only vs best multimodal ({best_caption_test})\n")
print(f"{'Class':<12} {'Baseline':>10} {'Multimodal':>12} {'Δ':>10} {'Rel %':>10}")
print('-' * 56)

for i, name in enumerate(CLASS_NAMES):
    base = unet_baseline_class_ious[i]
    mm   = best_results['class_ious'][i]
    if base is None or np.isnan(mm):
        print(f'{name:<12} {"N/A":>10} {"N/A":>12} {"":>10} {"":>10}')
        continue
    delta = mm - base
    rel = (delta / base * 100) if base > 0 else float('nan')
    print(f'{name:<12} {base:>10.4f} {mm:>12.4f} {delta:>+10.4f} {rel:>+9.1f}%')

print('-' * 56)
print(f'{"mIoU":<12} {unet_baseline_test_miou:>10.4f} {best_results["miou"]:>12.4f} '
      f'{best_results["miou"] - unet_baseline_test_miou:>+10.4f} '
      f'{(best_results["miou"] - unet_baseline_test_miou) / unet_baseline_test_miou * 100:>+9.1f}%')
print(f'{"OA":<12} {unet_baseline_test_oa:>10.4f} {best_results["oa"]:>12.4f} '
      f'{best_results["oa"] - unet_baseline_test_oa:>+10.4f} '
      f'{(best_results["oa"] - unet_baseline_test_oa) / unet_baseline_test_oa * 100:>+9.1f}%')

Per-class comparison: vision-only vs best multimodal (text_qwen3-4b)

Class          Baseline   Multimodal          Δ      Rel %
--------------------------------------------------------
Tree             0.8608       0.8758    +0.0150      +1.7%
Shrub            0.2476       0.3261    +0.0785     +31.7%
Grass            0.7631       0.8057    +0.0426      +5.6%
Crop             0.7416       0.8159    +0.0743     +10.0%
Built-up         0.5631       0.5893    +0.0262      +4.6%
Barren           0.4869       0.5665    +0.0796     +16.3%
Water            0.8784       0.8994    +0.0210      +2.4%
--------------------------------------------------------
mIoU             0.6488       0.6970    +0.0482      +7.4%
OA               0.8621       0.8894    +0.0274      +3.2%


**Observations.**

Per-class improvements concentrate on rare classes: Shrub (+31.7%), Barren (+16.3%), Crop (+10.0%). Dominant classes (Tree, Water) show smaller relative gains since the baseline already performs well on them. Built-up improves modestly (+4.6%), consistent with its low presence in the dataset (21.5% of images).

The pattern indicates that text augmentation is most beneficial precisely where the visual signal is weakest, for under-represented classes that the vision-only baseline tends to underperform on. This complements the overall mIoU improvement (+7.4%) by showing the model gains are not uniform but selective. Rare-class recall benefits the most.

## 7. Percentage-removal check

This section verifies that the multimodal benefit derives from qualitative semantic content rather than numerical class proportions in captions. The same training is repeated for the three text-rich captions using `_nopct` variants (numerical percentages removed, qualitative descriptors preserved). Vision-only captions are excluded since they do not contain percentages.

The expectation: if the benefit comes from qualitative content, removing numbers should not meaningfully change segmentation performance.

In [ ]:
# text_qwen3-4b WITHOUT percentages
hist_nopct, miou_nopct, ckpt_nopct = train_multimodal('text_qwen3-4b_nopct')
save_history(hist_nopct, 'multimodal_text_qwen3_4b_nopct')

/usr/local/lib/python3.12/dist-packages/timm/models/_factory.py:138: UserWarning: Mapping deprecated model name swsl_resnet18 to current resnet18.fb_swsl_ig1b_ft_in1k.
  model = create_fn(


Training multimodal_text_qwen3_4b_nopct for 30 epochs...
Epoch   T_Loss   V_Loss   V_mIoU     V_OA    Time       
-------------------------------------------------------
    1   1.1010   0.4367   0.5408   0.7752   58.0s   BEST
    2   0.7229   0.3953   0.5587   0.7855   58.6s   BEST
    3   0.6338   0.4681   0.5036   0.7802   57.5s       
    4   0.5843   0.3897   0.5203   0.7579   56.2s       
    5   0.5527   0.3990   0.5397   0.7636   56.6s       
    6   0.5213   0.3889   0.5574   0.7582   56.5s       
    7   0.5022   0.4124   0.5866   0.7790   57.4s   BEST
    8   0.5005   0.4042   0.5804   0.8050   56.1s       
    9   0.4747   0.2914   0.6029   0.8248   57.5s   BEST
   10   0.4433   0.3739   0.6337   0.8485   57.3s   BEST
   11   0.4327   0.2876   0.6085   0.8211   56.7s       
   12   0.4040   0.2672   0.6182   0.8405   56.7s       
   13   0.3878   0.2733   0.6369   0.8434   56.8s   BEST
   14   0.3795   0.2822   0.6214   0.8298   56.3s       
   15   0.3754   0.2763   0.6383

epoch,▁▁▁▂▂▂▂▃▃▃▃▄▄▄▄▅▅▅▅▆▆▆▆▇▇▇▇███
lr,█████▇▇▇▇▆▆▆▅▅▅▄▄▃▃▃▂▂▂▂▁▁▁▁▁▁
train_loss,█▅▄▄▃▃▃▃▃▃▂▂▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁
val_iou/Barren,▃▄▃▁▂▂▅▅▅▇▃▄▆▄▅▆▅█▆▅▇▇▇▇██▇█▇█
val_iou/Built-up,▃▃▅▁▄▅▆▇▇▇▆▅▇▆▇▆▆█▇▆▇▇▆█▇▇██▇█
val_iou/Crop,▂▄▃▄▁▃▅▄▄▆▅▆▆▄▅▆▇▇▇▇▇█▇███████
val_iou/Grass,▃▄▃▃▁▂▃▄▅▇▅▆▆▅▆▇▆▇▆▇▇▇▇▇██████
val_iou/Shrub,▂▂▅▂▄▁▁▃▄▃▄▅▄▄▄▄▃▄▅▇▆▆█▇▇█████
val_iou/Tree,▂▃▁▁▄▃▄▃▆▅▆▆▆▆▅▆▅▇▇▇▇▇▇███████
val_iou/Water,▆▇▁▇▆██▅▆▇█▇▇█████████████████
+3,...



Best validation mIoU: 0.7003
Saved history to /content/drive/MyDrive/DI725_Project/results/multimodal_text_qwen3_4b_nopct_history.json


In [ ]:
# hybrid_gemma3-4b WITHOUT percentages
hist_hg_nopct, miou_hg_nopct, ckpt_hg_nopct = train_multimodal('hybrid_gemma3-4b_nopct')
save_history(hist_hg_nopct, 'multimodal_hybrid_gemma3_4b_nopct')

/usr/local/lib/python3.12/dist-packages/timm/models/_factory.py:138: UserWarning: Mapping deprecated model name swsl_resnet18 to current resnet18.fb_swsl_ig1b_ft_in1k.
  model = create_fn(


Training multimodal_hybrid_gemma3_4b_nopct for 30 epochs...
Epoch   T_Loss   V_Loss   V_mIoU     V_OA    Time       
-------------------------------------------------------
    1   1.1302   0.4596   0.5478   0.7796   58.3s   BEST
    2   0.7673   0.4323   0.5226   0.7480   57.3s       
    3   0.6724   0.4042   0.5376   0.7809   57.0s       
    4   0.6055   0.3991   0.5418   0.7762   57.3s       
    5   0.5688   0.3538   0.5520   0.7894   56.9s   BEST
    6   0.5579   0.4036   0.4766   0.7472   56.8s       
    7   0.5163   0.3090   0.5909   0.8117   57.3s   BEST
    8   0.4870   0.3021   0.6159   0.8210   57.1s   BEST
    9   0.4733   0.2859   0.6102   0.8195   56.8s       
   10   0.4547   0.2887   0.6214   0.8241   57.6s   BEST
   11   0.4367   0.3264   0.5421   0.8055   57.2s       
   12   0.4136   0.2914   0.6377   0.8413   58.4s   BEST
   13   0.4104   0.2774   0.5947   0.8388   57.2s       
   14   0.3845   0.2788   0.6190   0.8365   57.1s       
   15   0.3688   0.2593   0.6

epoch,▁▁▁▂▂▂▂▃▃▃▃▄▄▄▄▅▅▅▅▆▆▆▆▇▇▇▇███
lr,█████▇▇▇▇▆▆▆▅▅▅▄▄▃▃▃▂▂▂▂▁▁▁▁▁▁
train_loss,█▅▄▄▃▃▃▃▃▃▂▂▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁
val_iou/Barren,▅▁▃▂▄▄▅▄▅▆▄▇▇▇▆▆▆▇▅█▇▅▇▇██████
val_iou/Built-up,▃▃▅▁▂▃▄▆▅▆▆▆▄▄▅▆██▇▇██▇▇▇▇██▇█
val_iou/Crop,▂▁▂▄▃▄▄▅▅▅▅▅▆▅▆▅▆▇▇██▆▇███████
val_iou/Grass,▂▁▃▃▂▃▄▄▄▅▅▅▆▆▅▅▇▇▆▇▇▆▇█▇▇████
val_iou/Shrub,▂▂▃▂▃▁▄▅▃▃▅▅▄▄▆▃▇▅▇▆▇▆█▇▇█████
val_iou/Tree,▄▄▄▅▅▁▅▇▆▆▄▇▆▇▇▆▇▇▇▇█▇████████
val_iou/Water,▆▆▄▇▆▁▇███▁█▅▇▇█▇████▇█▇██████
+3,...



Best validation mIoU: 0.6908
Saved history to /content/drive/MyDrive/DI725_Project/results/multimodal_hybrid_gemma3_4b_nopct_history.json


In [ ]:
# hybrid_qwen3-vl-8b WITHOUT percentages
hist_hq_nopct, miou_hq_nopct, ckpt_hq_nopct = train_multimodal('hybrid_qwen3-vl-8b_nopct')
save_history(hist_hq_nopct, 'multimodal_hybrid_qwen3_vl_8b_nopct')

Training multimodal_hybrid_qwen3_vl_8b_nopct for 30 epochs...
Epoch   T_Loss   V_Loss   V_mIoU     V_OA    Time       
-------------------------------------------------------
    1   1.1094   0.5830   0.4815   0.7385   57.7s   BEST
    2   0.7271   0.4078   0.5005   0.7582   58.3s   BEST
    3   0.6376   0.3959   0.5662   0.7816   58.0s   BEST
    4   0.5771   0.3812   0.5682   0.7942   57.9s   BEST
    5   0.5489   0.4771   0.5279   0.7674   57.5s       
    6   0.5137   0.3244   0.5731   0.8016   58.3s   BEST
    7   0.4802   0.3869   0.5953   0.7823   57.1s   BEST
    8   0.4635   0.2971   0.6239   0.8364   57.2s   BEST
    9   0.4604   0.2985   0.6348   0.8403   57.4s   BEST
   10   0.4452   0.2943   0.6333   0.8426   56.5s       
   11   0.4087   0.2917   0.6291   0.8397   56.8s       
   12   0.4049   0.2606   0.6299   0.8425   56.9s       
   13   0.3875   0.2844   0.6400   0.8407   58.2s   BEST
   14   0.3843   0.2561   0.6261   0.8356   56.8s       
   15   0.3571   0.2512   0

epoch,▁▁▁▂▂▂▂▃▃▃▃▄▄▄▄▅▅▅▅▆▆▆▆▇▇▇▇███
lr,█████▇▇▇▇▆▆▆▅▅▅▄▄▃▃▃▂▂▂▂▁▁▁▁▁▁
train_loss,█▅▄▄▃▃▃▃▃▃▂▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁
val_iou/Barren,▁▁▂▄▂▁▅▅▆▅▃▅▆▄▄▆▅▆▅▅▃▅▇▇█▇▇█▇▇
val_iou/Built-up,▄▃▇▆▁▆█▆▇█▇▆▇▇▇▇▇█▇▇▇▇██▇▇████
val_iou/Crop,▃▂▂▁▁▄▆▅▆▄▆▆▆▆▆▆▆▇▆▇▇██▇██████
val_iou/Grass,▁▃▃▄▃▄▃▆▆▆▆▆▆▆▆▇▇▇▇▇▇▇▇▇██████
val_iou/Shrub,▂▂▃▄▅▄▁▃▅▆▆▅▅▄▄▆▆▄▆▆▆▆▇▆▇█▇▇▇▇
val_iou/Tree,▁▃▅▅▂▆▃▆▆▇▇▇▆▇▇▇▇▇▆▇▇█████████
val_iou/Water,▁▂▅▄▆▅▇█▇▆▇▇█▇▇█▇█████████████
+3,...



Best validation mIoU: 0.6914
Saved history to /content/drive/MyDrive/DI725_Project/results/multimodal_hybrid_qwen3_vl_8b_nopct_history.json


### Test set evaluation

Evaluate the 3 no-percentage checkpoints on the test set. Their results are appended to `multimodal_test_results` (which already contains the 5 originals from Section 6) so that the next table can compare each variant pair directly.

In [ ]:
# Evaluate each no-percentage checkpoint on the test set
nopct_runs = [
    ('text_qwen3-4b_nopct',       CHECKPOINTS_DIR / 'multimodal_text_qwen3_4b_nopct_best.pth'),
    ('hybrid_gemma3-4b_nopct',    CHECKPOINTS_DIR / 'multimodal_hybrid_gemma3_4b_nopct_best.pth'),
    ('hybrid_qwen3-vl-8b_nopct',  CHECKPOINTS_DIR / 'multimodal_hybrid_qwen3_vl_8b_nopct_best.pth'),
]

# Append nopct results to the multimodal_test_results dict (already contains 5 originals)
for caption_col, ckpt_path in nopct_runs:
    print(f'Evaluating {caption_col}...')
    class_ious, miou, oa, loss = evaluate_checkpoint(caption_col, ckpt_path)
    multimodal_test_results[caption_col] = {
        'class_ious': class_ious,
        'miou':       miou,
        'oa':         oa,
        'loss':       loss,
    }
    print(f'  test mIoU: {miou:.4f}, OA: {oa:.4f}')

Evaluating text_qwen3-4b_nopct...
  test mIoU: 0.6987, OA: 0.8895
Evaluating hybrid_gemma3-4b_nopct...
  test mIoU: 0.6874, OA: 0.8838
Evaluating hybrid_qwen3-vl-8b_nopct...
  test mIoU: 0.6855, OA: 0.8829


### Percentage-removal results

Direct comparison of each text-rich caption variant with and without numerical percentages.

In [ ]:
# Methodological ablation: with vs without numerical percentages
print(f"\nMethodological ablation: does the multimodal benefit depend on percentages?\n")
print(f"{'Variant':<28} {'With %':>10} {'Without %':>12} {'Δ':>10} {'Rel %':>10}")
print('-' * 75)

ABLATION_PAIRS = [
    ('text_qwen3-4b',       'text_qwen3-4b_nopct'),
    ('hybrid_gemma3-4b',    'hybrid_gemma3-4b_nopct'),
    ('hybrid_qwen3-vl-8b',  'hybrid_qwen3-vl-8b_nopct'),
]

for orig, nopct in ABLATION_PAIRS:
    m_orig  = multimodal_test_results[orig]['miou']
    m_nopct = multimodal_test_results[nopct]['miou']
    delta   = m_nopct - m_orig
    rel     = delta / m_orig * 100
    print(f'{orig:<28} {m_orig:>10.4f} {m_nopct:>12.4f} {delta:>+10.4f} {rel:>+9.2f}%')


Methodological ablation: does the multimodal benefit depend on percentages?

Variant                          With %    Without %          Δ      Rel %
---------------------------------------------------------------------------
text_qwen3-4b                    0.6970       0.6987    +0.0017     +0.25%
hybrid_gemma3-4b                 0.6930       0.6874    -0.0056     -0.81%
hybrid_qwen3-vl-8b               0.6906       0.6855    -0.0051     -0.73%


**Observations**

All three deltas fall within ±0.81%, within typical run-to-run variance for this training setup. Removing numerical percentages from captions does not meaningfully change segmentation performance. The multimodal benefit therefore comes from qualitative semantic content rather than the numerical class proportions present in some captions.

In [ ]:
# Persist multimodal test results alongside baseline for downstream analysis

ORIGINAL_CAPTIONS = [
    'hybrid_gemma3-4b',
    'hybrid_qwen3-vl-8b',
    'text_qwen3-4b',
    'vision_gemma3-4b',
    'vision_qwen3-vl-8b',
]

# Best caption among the 5 originals (used by downstream KL pipeline)
best_caption_test = max(
    ORIGINAL_CAPTIONS,
    key=lambda k: multimodal_test_results[k]['miou']
)
best_caption = best_caption_test

multimodal_results_to_save = {
    'baseline_unetformer': {
        'test_miou':  unet_baseline_test_miou,
        'test_oa':    unet_baseline_test_oa,
        'class_ious': unet_baseline_class_ious,
    },
    'multimodal': {
        cap: {
            'test_miou':  res['miou'],
            'test_oa':    res['oa'],
            'test_loss':  res['loss'],
            'class_ious': [float(x) if not np.isnan(x) else None for x in res['class_ious']],
        }
        for cap, res in multimodal_test_results.items()
        if not cap.endswith('_nopct')  # exclude ablation variants from main results
    },
    'percentage_ablation': {
        cap: {
            'test_miou':  res['miou'],
            'test_oa':    res['oa'],
            'test_loss':  res['loss'],
            'class_ious': [float(x) if not np.isnan(x) else None for x in res['class_ious']],
        }
        for cap, res in multimodal_test_results.items()
        if cap.endswith('_nopct')  # only ablation variants
    },
    'class_names':       CLASS_NAMES,
    'best_caption_val':  best_caption,
    'best_caption_test': best_caption_test,
}

results_path = RESULTS_DIR / 'multimodal_results.json'
with open(results_path, 'w') as f:
    json.dump(multimodal_results_to_save, f, indent=2)
print(f'Saved to {results_path}')
print(f'Best caption (among 5 originals): {best_caption_test}')

Saved to /content/drive/MyDrive/DI725_Project/results/multimodal_results.json
Best caption (among 5 originals): text_qwen3-4b


## 8. KL-divergence auxiliary loss ablation

Take the best caption variant from the test evaluation (`text_qwen3-4b`) and add a KL-divergence based train-time auxiliary regularizer between the predicted class distribution (spatial mean of softmax logits) and the image-level class distribution from the captions CSV. The total loss becomes:

`total_loss = main_ce + 0.4 × aux_ce + kl_weight × kl(gt_dist || pred_dist)`

The auxiliary regularizer is used only during training and is not provided to the model during inference. Test mIoU is reported with `kl_weight = 0.5`, the value selected from preliminary tuning, to evaluate whether distribution-level supervision provides additional benefit on top of textual augmentation.

In [ ]:
class RSMultiModalDatasetWithDist(Dataset):
    """Same as RSMultiModalDataset but additionally returns the CSV-derived
    image-level class distribution per image (used for the KL auxiliary regularizer)."""

    def __init__(self, dataframe, images_dir, masks_dir, caption_col):
        self.df = dataframe.reset_index(drop=True)
        self.images_dir = Path(images_dir)
        self.masks_dir  = Path(masks_dir)
        self.caption_col = caption_col
        self.normalize = transforms.Normalize(
            mean=[0.485, 0.456, 0.406],
            std=[0.229, 0.224, 0.225],
        )

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        fname = row['filename']

        img = Image.open(self.images_dir / fname).convert('RGB')
        img = transforms.functional.to_tensor(img)
        img = self.normalize(img)

        mask = np.array(Image.open(self.masks_dir / fname))
        mask = torch.from_numpy(mask).long()

        caption = str(row[self.caption_col])

        # Ground-truth class distribution from CSV percentages (sum ≈ 1.0)
        gt_dist = torch.tensor(
            [row[c] / 100.0 for c in CLASS_NAMES],
            dtype=torch.float32,
        )

        return img, mask, caption, gt_dist


def make_loaders_with_dist(caption_col):
    train_dataset = RSMultiModalDatasetWithDist(train_df, LOCAL_IMAGES, LOCAL_MASKS_CLASS, caption_col)
    val_dataset   = RSMultiModalDatasetWithDist(val_df,   LOCAL_IMAGES, LOCAL_MASKS_CLASS, caption_col)
    test_dataset  = RSMultiModalDatasetWithDist(test_df,  LOCAL_IMAGES, LOCAL_MASKS_CLASS, caption_col)

    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,
                              num_workers=NUM_WORKERS, pin_memory=True)
    val_loader   = DataLoader(val_dataset,  batch_size=BATCH_SIZE, shuffle=False,
                              num_workers=NUM_WORKERS, pin_memory=True)
    test_loader  = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False,
                              num_workers=NUM_WORKERS, pin_memory=True)
    return train_loader, val_loader, test_loader

In [ ]:
def train_one_epoch_kl_csv(model, loader, optimizer, criterion, device, kl_weight):
    """Train one epoch with KL auxiliary regularizer using CSV-derived class distribution."""
    model.train()
    total_loss = 0
    for imgs, masks, captions, gt_dists in loader:
        imgs, masks, gt_dists = imgs.to(device), masks.to(device), gt_dists.to(device)
        logits_main, logits_aux = model(imgs, list(captions))

        loss = criterion(logits_main, masks) + AUX_LOSS_WEIGHT * criterion(logits_aux, masks)

        # KL term: predicted class distribution vs CSV-derived ground truth
        pred_dist = F.softmax(logits_main, dim=1).mean(dim=[2, 3])
        kl_loss = F.kl_div(
            torch.log(pred_dist + 1e-8), gt_dists + 1e-8,
            reduction='batchmean',
        )
        loss = loss + kl_weight * kl_loss

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    return total_loss / len(loader)


@torch.no_grad()
def evaluate_with_dist(model, loader, criterion, device, num_classes=NUM_CLASSES):
    """Evaluate; ignores the gt_dist tuple element (only used during training).
    Returns: per-class pixel IoU list, pixel-level mIoU, overall accuracy, average loss."""
    model.eval()
    intersection = np.zeros(num_classes, dtype=np.int64)
    union        = np.zeros(num_classes, dtype=np.int64)
    total_correct = 0
    total_pixels  = 0
    total_loss    = 0
    for imgs, masks, captions, _ in loader:
        imgs, masks = imgs.to(device), masks.to(device)
        logits = model(imgs, list(captions))
        total_loss += criterion(logits, masks).item()
        preds = logits.argmax(dim=1)
        update_confusion(intersection, union, preds, masks, num_classes)
        total_correct += (preds == masks).sum().item()
        total_pixels  += masks.numel()
    class_ious, miou = finalize_iou(intersection, union, num_classes)
    oa = total_correct / total_pixels
    return class_ious, miou, oa, total_loss / len(loader)

In [ ]:
def train_multimodal_kl_csv(caption_col, kl_weight, num_epochs=NUM_EPOCHS, lr=LR,
                            run_suffix=None):
    """Train UNetFormerCLIP with KL auxiliary regularizer using CSV-derived class distribution."""
    base_name = caption_col.replace('-', '_').replace('/', '_')
    suffix = run_suffix or f'kl_{int(kl_weight * 10):02d}'
    run_name = f'multimodal_{base_name}_{suffix}'
    save_path = CHECKPOINTS_DIR / f'{run_name}_best.pth'

    train_loader, val_loader, _ = make_loaders_with_dist(caption_col)

    model = UNetFormerCLIP(num_classes=NUM_CLASSES).to(device)
    trainable = filter(lambda p: p.requires_grad, model.parameters())
    optimizer = AdamW(trainable, lr=lr, weight_decay=WEIGHT_DECAY)
    scheduler = CosineAnnealingLR(optimizer, T_max=num_epochs, eta_min=1e-6)
    criterion = nn.CrossEntropyLoss(weight=class_weights_tensor)

    run = wandb.init(
        project=WANDB_PROJECT,
        name=run_name,
        config={
            'model':           'UNetFormerCLIP',
            'caption_col':     caption_col,
            'kl_weight':       kl_weight,
            'kl_source':       'csv',
            'num_epochs':      num_epochs,
            'lr':              lr,
            'weight_decay':    WEIGHT_DECAY,
            'batch_size':      BATCH_SIZE,
            'aux_loss_weight': AUX_LOSS_WEIGHT,
            'seed':            SEED,
        },
        reinit=True,
    )

    history = {'train_loss': [], 'val_loss': [], 'val_miou': [], 'val_oa': []}
    best_miou = 0.0

    print(f'Training {run_name} for {num_epochs} epochs...')
    print(f"{'Epoch':>5} {'T_Loss':>8} {'V_Loss':>8} {'V_mIoU':>8} {'V_OA':>8} {'Time':>7} {'':>6}")
    print('-' * 55)

    for epoch in range(num_epochs):
        t0 = time.time()
        train_loss = train_one_epoch_kl_csv(model, train_loader, optimizer, criterion, device, kl_weight)
        val_class_ious, val_miou, val_oa, val_loss = evaluate_with_dist(model, val_loader, criterion, device)
        scheduler.step()

        history['train_loss'].append(train_loss)
        history['val_loss'].append(val_loss)
        history['val_miou'].append(val_miou)
        history['val_oa'].append(val_oa)

        log_dict = {
            'epoch':      epoch + 1,
            'train_loss': train_loss,
            'val_loss':   val_loss,
            'val_miou':   val_miou,
            'val_oa':     val_oa,
            'lr':         scheduler.get_last_lr()[0],
        }
        for c, name in enumerate(CLASS_NAMES):
            if not np.isnan(val_class_ious[c]):
                log_dict[f'val_iou/{name}'] = val_class_ious[c]
        wandb.log(log_dict)

        status = ''
        if val_miou > best_miou:
            best_miou = val_miou
            torch.save(model.state_dict(), str(save_path))
            status = 'BEST'

        elapsed = time.time() - t0
        print(f'{epoch+1:>5} {train_loss:>8.4f} {val_loss:>8.4f} {val_miou:>8.4f} {val_oa:>8.4f} {elapsed:>6.1f}s {status:>6}')

    wandb.summary['best_val_miou'] = best_miou
    wandb.finish()
    print(f'\nBest validation mIoU: {best_miou:.4f}')
    return history, best_miou, save_path

In [ ]:
# KL ablation on the best caption variant from test eval (text_qwen3-4b)
KL_CAPTION = 'text_qwen3-4b'
kl_weights = [0.5]
kl_runs = {}

for w in kl_weights:
    suffix = f'kl_{int(w * 10):02d}'
    print(f'\n{"=" * 60}')
    print(f'Caption: {KL_CAPTION}, KL weight = {w}')
    print(f'{"=" * 60}\n')
    hist, miou, ckpt = train_multimodal_kl_csv(
        KL_CAPTION, kl_weight=w, run_suffix=suffix
    )
    base_name = KL_CAPTION.replace('-', '_').replace('/', '_')
    save_history(hist, f'multimodal_{base_name}_{suffix}')
    kl_runs[w] = {'history': hist, 'best_val_miou': miou, 'ckpt': ckpt}

print('\n' + '=' * 60)
print('KL sweep complete')
print('=' * 60)
for w, run_info in kl_runs.items():
    print(f'  caption={KL_CAPTION}, weight={w}: best val mIoU = {run_info["best_val_miou"]:.4f}')


Caption: text_qwen3-4b, KL weight = 0.5



wandb: WARNING Using a boolean value for 'reinit' is deprecated. Use 'return_previous' or 'finish_previous' instead.


Training multimodal_text_qwen3_4b_kl_05 for 30 epochs...
Epoch   T_Loss   V_Loss   V_mIoU     V_OA    Time       
-------------------------------------------------------
    1   1.3465   0.4911   0.5338   0.7727   58.7s   BEST
    2   0.8537   0.4566   0.5644   0.8141   58.4s   BEST
    3   0.7744   0.3969   0.5833   0.8119   58.0s   BEST
    4   0.6892   0.4047   0.5245   0.8076   56.8s       
    5   0.6362   0.4290   0.5590   0.8055   56.6s       
    6   0.6105   0.4829   0.5696   0.7827   58.1s       
    7   0.5587   0.3051   0.6411   0.8566   58.2s   BEST
    8   0.5457   0.3260   0.6204   0.8509   57.3s       
    9   0.5234   0.2933   0.6358   0.8560   58.3s       
   10   0.4825   0.3014   0.6299   0.8445   56.5s       
   11   0.4938   0.3606   0.6074   0.8160   57.4s       
   12   0.4658   0.2861   0.6414   0.8505   57.4s   BEST
   13   0.4331   0.2586   0.6531   0.8632   57.6s   BEST
   14   0.4428   0.2957   0.6454   0.8515   58.1s       
   15   0.4109   0.2668   0.6475

epoch,▁▁▁▂▂▂▂▃▃▃▃▄▄▄▄▅▅▅▅▆▆▆▆▇▇▇▇███
lr,█████▇▇▇▇▆▆▆▅▅▅▄▄▃▃▃▂▂▂▂▁▁▁▁▁▁
train_loss,█▅▄▄▃▃▃▃▃▂▂▂▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁
val_iou/Barren,▁▄▁▃▁▄▆▆▆▂▁▃▅▅▅▇▄▇▆▇▅▇▇▇▇▇▇███
val_iou/Built-up,▃▁▅▅▅▇▅▄▅▆▇▇▆█▇▇█▇▇▇▇▇▇█▇█████
val_iou/Crop,▁▂▂▃▃▁▅▅▅▅▄▆▆▄▆▇▇▇▇▇▇▇████████
val_iou/Grass,▁▄▃▅▃▁▆▆▆▅▅▅▇▆▆▇▇▇▇▇▇█▇███████
val_iou/Shrub,▁▃▅▅▃▂▆▅▅▅▃▆▄▄▅▆▇▅█▆▆▇▆▇▇█▇██▇
val_iou/Tree,▂▃▅▁▄▁▆▅▇▇▂▆▆▆▇▇▇▆█▇▇█████████
val_iou/Water,▆▆▆▁▅▆▇▇▇▇▇▇█▇▇███████████████
+3,...



Best validation mIoU: 0.7089
Saved history to /content/drive/MyDrive/DI725_Project/results/multimodal_text_qwen3_4b_kl_05_history.json

KL sweep complete
  caption=text_qwen3-4b, weight=0.5: best val mIoU = 0.7089


In [ ]:
def evaluate_checkpoint_with_dist(caption_col, ckpt_path):
    """Evaluate a multimodal checkpoint on the test set.

    Uses make_loaders_with_dist to read CSV class distributions,
    matching the data layout used during KL-augmented training."""
    _, _, test_loader = make_loaders_with_dist(caption_col)

    model = UNetFormerCLIP(num_classes=NUM_CLASSES).to(device)
    model.load_state_dict(torch.load(str(ckpt_path)))
    criterion = nn.CrossEntropyLoss(weight=class_weights_tensor)

    class_ious, miou, oa, loss = evaluate_with_dist(model, test_loader, criterion, device)
    del model
    torch.cuda.empty_cache()
    return class_ious, miou, oa, loss


# Test eval for each KL run (using the caption set in cell 47)
for w, run_info in kl_runs.items():
    print(f'Evaluating KL weight = {w} on test set...')
    class_ious, test_miou, test_oa, test_loss = evaluate_checkpoint_with_dist(
        KL_CAPTION, run_info['ckpt']
    )
    run_info['test_class_ious'] = class_ious
    run_info['test_miou']       = test_miou
    run_info['test_oa']         = test_oa
    run_info['test_loss']       = test_loss
    print(f'  test mIoU: {test_miou:.4f}, OA: {test_oa:.4f}')

Evaluating KL weight = 0.5 on test set...
  test mIoU: 0.7046, OA: 0.8932


In [ ]:
# Summary: KL weight vs test mIoU vs vision-only and best multimodal
best_no_kl = multimodal_test_results[KL_CAPTION]['miou']

print(f"{'Configuration':<30} {'Test mIoU':>10} {'Δ vs no-KL':>12} {'Rel %':>9} {'Δ vs base':>12} {'Rel %':>9}")
print('-' * 85)

# vision-only
print(f'{"Vision-only baseline":<30} {unet_baseline_test_miou:>10.4f} '
      f'{"--":>12} {"--":>9} {"--":>12} {"--":>9}')

# no-KL multimodal
delta_b = best_no_kl - unet_baseline_test_miou
rel_b = delta_b / unet_baseline_test_miou * 100
print(f'{"Multimodal (no KL)":<30} {best_no_kl:>10.4f} '
      f'{"--":>12} {"--":>9} {delta_b:>+12.4f} {rel_b:>+8.1f}%')

# KL rows
for w, run_info in kl_runs.items():
    delta_no_kl = run_info['test_miou'] - best_no_kl
    rel_no_kl = delta_no_kl / best_no_kl * 100
    delta_b = run_info['test_miou'] - unet_baseline_test_miou
    rel_b = delta_b / unet_baseline_test_miou * 100
    label = f'Multimodal + KL (w={w})'
    print(f'{label:<30} {run_info["test_miou"]:>10.4f} '
          f'{delta_no_kl:>+12.4f} {rel_no_kl:>+8.1f}% '
          f'{delta_b:>+12.4f} {rel_b:>+8.1f}%')

best_kl_weight = max(kl_runs, key=lambda w: kl_runs[w]['test_miou'])
best_kl_test = kl_runs[best_kl_weight]['test_miou']
print(f'\nBest KL configuration: caption={KL_CAPTION}, weight = {best_kl_weight} → test mIoU = {best_kl_test:.4f}')

Configuration                   Test mIoU   Δ vs no-KL     Rel %    Δ vs base     Rel %
-------------------------------------------------------------------------------------
Vision-only baseline               0.6488           --        --           --        --
Multimodal (no KL)                 0.6970           --        --      +0.0482     +7.4%
Multimodal + KL (w=0.5)            0.7046      +0.0076     +1.1%      +0.0558     +8.6%

Best KL configuration: caption=text_qwen3-4b, weight = 0.5 → test mIoU = 0.7046


**Observations.**

Adding the KL-divergence auxiliary regularizer to `text_qwen3-4b` improves test mIoU by +1.1% relative (0.6970 to 0.7046), bringing the total gain over the vision-only baseline to +8.6%. The +1.1% magnitude is consistent with KL acting as a complementary training-time regularizer alongside the weighted cross-entropy and auxiliary head losses. The auxiliary signal is used only during training, so test mIoU reflects what the model learned from image and caption inputs alone.

In [ ]:
# Per-class comparison: vision-only vs best multimodal vs best KL config
best_kl_run = kl_runs[best_kl_weight]
best_no_kl_results = multimodal_test_results[KL_CAPTION]

print(f'Per-class: vision-only vs {KL_CAPTION} vs +KL (weight={best_kl_weight})\n')
print(f"{'Class':<11} {'Baseline':>10} {'Multimodal':>12} {'+KL':>10} "
      f"{'Δ MM':>9} {'Rel % MM':>10} {'Δ KL':>9} {'Rel % KL':>10} "
      f"{'Δ Total':>10} {'Rel % Total':>12}")
print('-' * 115)

for i, name in enumerate(CLASS_NAMES):
    base = unet_baseline_class_ious[i]
    mm   = best_no_kl_results['class_ious'][i]
    kl   = best_kl_run['test_class_ious'][i]
    if base is None or np.isnan(mm) or np.isnan(kl):
        print(f'{name:<11} {"N/A":>10} {"N/A":>12} {"N/A":>10}')
        continue
    rel_mm    = (mm - base) / base * 100 if base > 0 else float('nan')
    rel_kl    = (kl - mm) / mm * 100 if mm > 0 else float('nan')
    rel_total = (kl - base) / base * 100 if base > 0 else float('nan')
    print(f'{name:<11} {base:>10.4f} {mm:>12.4f} {kl:>10.4f} '
          f'{mm-base:>+9.4f} {rel_mm:>+9.1f}% '
          f'{kl-mm:>+9.4f} {rel_kl:>+9.1f}% '
          f'{kl-base:>+10.4f} {rel_total:>+11.1f}%')

print('-' * 115)

# mIoU
mm_miou = best_no_kl_results['miou']
kl_miou = best_kl_run['test_miou']
rel_mm_miou    = (mm_miou - unet_baseline_test_miou) / unet_baseline_test_miou * 100
rel_kl_miou    = (kl_miou - mm_miou) / mm_miou * 100
rel_total_miou = (kl_miou - unet_baseline_test_miou) / unet_baseline_test_miou * 100
print(f'{"mIoU":<11} {unet_baseline_test_miou:>10.4f} {mm_miou:>12.4f} {kl_miou:>10.4f} '
      f'{mm_miou - unet_baseline_test_miou:>+9.4f} {rel_mm_miou:>+9.1f}% '
      f'{kl_miou - mm_miou:>+9.4f} {rel_kl_miou:>+9.1f}% '
      f'{kl_miou - unet_baseline_test_miou:>+10.4f} {rel_total_miou:>+11.1f}%')

# OA
mm_oa = best_no_kl_results['oa']
kl_oa = best_kl_run['test_oa']
rel_mm_oa    = (mm_oa - unet_baseline_test_oa) / unet_baseline_test_oa * 100
rel_kl_oa    = (kl_oa - mm_oa) / mm_oa * 100
rel_total_oa = (kl_oa - unet_baseline_test_oa) / unet_baseline_test_oa * 100
print(f'{"OA":<11} {unet_baseline_test_oa:>10.4f} {mm_oa:>12.4f} {kl_oa:>10.4f} '
      f'{mm_oa - unet_baseline_test_oa:>+9.4f} {rel_mm_oa:>+9.1f}% '
      f'{kl_oa - mm_oa:>+9.4f} {rel_kl_oa:>+9.1f}% '
      f'{kl_oa - unet_baseline_test_oa:>+10.4f} {rel_total_oa:>+11.1f}%')

Per-class: vision-only vs text_qwen3-4b vs +KL (weight=0.5)

Class         Baseline   Multimodal        +KL      Δ MM   Rel % MM      Δ KL   Rel % KL    Δ Total  Rel % Total
-------------------------------------------------------------------------------------------------------------------
Tree            0.8608       0.8758     0.8744   +0.0150      +1.7%   -0.0015      -0.2%    +0.0136        +1.6%
Shrub           0.2476       0.3261     0.3515   +0.0785     +31.7%   +0.0254      +7.8%    +0.1039       +42.0%
Grass           0.7631       0.8057     0.8125   +0.0426      +5.6%   +0.0068      +0.8%    +0.0494        +6.5%
Crop            0.7416       0.8159     0.8181   +0.0743     +10.0%   +0.0022      +0.3%    +0.0765       +10.3%
Built-up        0.5631       0.5893     0.6212   +0.0262      +4.6%   +0.0319      +5.4%    +0.0580       +10.3%
Barren          0.4869       0.5665     0.5809   +0.0796     +16.3%   +0.0144      +2.5%    +0.0939       +19.3%
Water           0.8784       0.8

**Observations**

The KL contribution is selective: Built-up (+5.4%) and Shrub (+7.8%) gain the most, while dominant classes show marginal or slightly negative changes (Water -2.9%, Tree -0.2%). KL helps most where the multimodal model still struggles, complementing rather than overlapping with the text contribution. The total gain over the vision-only baseline reaches +42.0% on Shrub and +19.3% on Barren, the two rarest classes in the dataset.

In [ ]:
# Update the multimodal results JSON with KL ablation
with open(RESULTS_DIR / 'multimodal_results.json') as f:
    results = json.load(f)

results['kl_ablation'] = {
    'caption_col': KL_CAPTION,
    'kl_source':   'csv',
    'runs': {
        f'kl_{w}': {
            'kl_weight':     w,
            'val_miou_best': float(run_info['best_val_miou']),
            'test_miou':     float(run_info['test_miou']),
            'test_oa':       float(run_info['test_oa']),
            'test_loss':     float(run_info['test_loss']),
            'class_ious':    [float(x) if not np.isnan(x) else None for x in run_info['test_class_ious']],
        }
        for w, run_info in kl_runs.items()
    },
    'best_kl_weight': float(best_kl_weight),
}

with open(RESULTS_DIR / 'multimodal_results.json', 'w') as f:
    json.dump(results, f, indent=2)
print(f'Updated {RESULTS_DIR / "multimodal_results.json"}')

Updated /content/drive/MyDrive/DI725_Project/results/multimodal_results.json


## Summary

Three experiments evaluated multimodal fusion for remote sensing land cover segmentation, all using pixel-level mIoU computed by accumulating intersection and union counts across the test set.

**Caption-variant ablation.** Among the 5 caption variants, `text_qwen3-4b` reached the highest mIoU. Hybrid captions performed comparably, while vision-only captions contributed negligibly. Caption quality, driven by the inclusion of class-aware textual content, drives the multimodal benefit.

**Percentage-removal check.** Stripping numerical percentages from the three text-rich caption variants changed mIoU by less than ±0.81% in all cases, within run-to-run variance. The multimodal benefit derives from qualitative semantic content, not numerical class proportions.

**KL-divergence auxiliary regularizer.** Adding the KL term during training for `text_qwen3-4b` further improved test mIoU by +1.1% relative.

The final configuration (`text_qwen3-4b` + KL with weight 0.5) recovers +42.0% on Shrub and +19.3% on Barren over the vision-only baseline, the two rarest classes in the dataset. Detailed results are saved to `multimodal_results.json` for downstream analysis.